In [1]:
# Import the necessary libraries and packages.
import pandas as pd
import os
import re
import unicodedata
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, insert
from sqlalchemy.orm import declarative_base, relationship, Session
from sqlalchemy import text, Column, Integer, BigInteger, String, Date, TIMESTAMP, ForeignKey, Index, CheckConstraint, UniqueConstraint

In [2]:
# Adjust the display options.
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_colwidth", None)

In [3]:
# # Reset the display options.
# pd.reset_option("display.max_rows")
# pd.reset_option("display.max_colwidth")

In [4]:
# load .env from the project's root folder.
load_dotenv()

True

In [5]:
# PostgreSQL database needed info.
USERNAME = os.getenv("DB_USERNAME")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
ORIGINAL_DB_NAME = os.getenv("ORIGINAL_DB_NAME")
NORMALIZED_DB_NAME = os.getenv("NORMALIZED_DB_NAME")

# To create the engine that establishes the connection with the original database, we type:
engine_original_db = create_engine(f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{ORIGINAL_DB_NAME}")

# To create the engine that establishes the connection with the normalized database, we type:
engine_normalized_db = create_engine(f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{NORMALIZED_DB_NAME}")

# Create the session.
session = Session(engine_normalized_db)

In [6]:
# Uncomment the following lines of code, to test the connection to the databases.
# with engine_original_db.connect() as conn:
#     result = conn.execute(text("SELECT version();"))
#     print(result.fetchone())

# with engine_normalized_db.connect() as conn:
#     result = conn.execute(text("SELECT version();"))
#     print(result.fetchone())

In [7]:
# Store the "genre" entity into a new DataFrame.
df_original_db_genre_table = pd.read_sql("SELECT * FROM genre", engine_original_db)

# Define the column datatypes.
schema = {
    "genre_id": "Int64",
    "name": "string"
}

# Apply the datatypes.
df_original_db_genre_table = df_original_db_genre_table.astype(schema)

print(df_original_db_genre_table)
df_original_db_genre_table.info()

    genre_id                name
0          1                Rock
1          2                Jazz
2          3               Metal
3          4  Alternative & Punk
4          5       Rock And Roll
5          6               Blues
6          7               Latin
7          8              Reggae
8          9                 Pop
9         10          Soundtrack
10        11          Bossa Nova
11        12      Easy Listening
12        13         Heavy Metal
13        14            R&B/Soul
14        15   Electronica/Dance
15        16               World
16        17         Hip Hop/Rap
17        18     Science Fiction
18        19            TV Shows
19        20    Sci Fi & Fantasy
20        21               Drama
21        22              Comedy
22        23         Alternative
23        24           Classical
24        25               Opera
<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------  

**As we can see above, we have a couple genres in the database, that should be merged.<br>**
**Specifically the<br>**
**_Alternative & Punk_ and _Alternative_<br>**
**and<br>**
**_Science Fiction_ and _Sci Fi & Fantasy_ genres.<br>**
**We start by removing the rows that contain the _Alternative_ and _Science Fiction_ genres.**

In [8]:
df_original_db_genre_table.loc[[17, 22]]

,genre_id,name
17,18,Science Fiction
22,23,Alternative


In [9]:
# We use ".drop()" alongside the index of the rows we want to remove.
df_original_db_genre_table.drop([17, 22], inplace=True)
print(df_original_db_genre_table)

    genre_id                name
0          1                Rock
1          2                Jazz
2          3               Metal
3          4  Alternative & Punk
4          5       Rock And Roll
5          6               Blues
6          7               Latin
7          8              Reggae
8          9                 Pop
9         10          Soundtrack
10        11          Bossa Nova
11        12      Easy Listening
12        13         Heavy Metal
13        14            R&B/Soul
14        15   Electronica/Dance
15        16               World
16        17         Hip Hop/Rap
18        19            TV Shows
19        20    Sci Fi & Fantasy
20        21               Drama
21        22              Comedy
23        24           Classical
24        25               Opera


In [10]:
# Store the "track" entity into a new DataFrame.
df_original_db_track_table = pd.read_sql("SELECT * FROM track", engine_original_db)

schema = {
    "track_id": "Int64",
    "name": "string",
    "album_id": "Int64",
    "media_type_id": "Int64",
    "genre_id": "Int64",
    "composer": "string",
    "milliseconds": "Int64",
    "bytes": "Int64",
    "unit_price": "Float64"
}

df_original_db_track_table = df_original_db_track_table.astype(schema)

df_original_db_track_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 3503 entries, 0 to 3502
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   track_id       3503 non-null   Int64  
 1   name           3503 non-null   string 
 2   album_id       3503 non-null   Int64  
 3   media_type_id  3503 non-null   Int64  
 4   genre_id       3503 non-null   Int64  
 5   composer       2525 non-null   string 
 6   milliseconds   3503 non-null   Int64  
 7   bytes          3503 non-null   Int64  
 8   unit_price     3503 non-null   Float64
dtypes: Float64(1), Int64(6), string(2)
memory usage: 386.2 KB


In [11]:
# We replace the genre IDs of the deleted genres (Alternative and Science Fiction), concluding the process of merging the genres mentioned previously.
df_original_db_track_table.loc[df_original_db_track_table["genre_id"] == 23, "genre_id"] = 4
df_original_db_track_table.loc[df_original_db_track_table["genre_id"] == 18, "genre_id"] = 20

In [12]:
# Check the "track" DataFrame.
df_original_db_track_table.head()

,track_id,name,album_id,media_type_id,genre_id,composer,milliseconds,bytes,unit_price
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,<NA>,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99
3,4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051,4331779,0.99
4,5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99


**Move on to the process of building the transitional mapping table that
will be used to create the "genre_track" junction table in the normalized database.**

In [13]:
# Using the ".split()" method and regex, we split each string of the "name" column when we meet a " & "or "/" symbol.
df_original_db_genre_table["name_split"] = df_original_db_genre_table["name"].str.split(r"\s&\s|/")

In [14]:
print(df_original_db_genre_table)

    genre_id                name            name_split
0          1                Rock                [Rock]
1          2                Jazz                [Jazz]
2          3               Metal               [Metal]
3          4  Alternative & Punk   [Alternative, Punk]
4          5       Rock And Roll       [Rock And Roll]
5          6               Blues               [Blues]
6          7               Latin               [Latin]
7          8              Reggae              [Reggae]
8          9                 Pop                 [Pop]
9         10          Soundtrack          [Soundtrack]
10        11          Bossa Nova          [Bossa Nova]
11        12      Easy Listening      [Easy Listening]
12        13         Heavy Metal         [Heavy Metal]
13        14            R&B/Soul           [R&B, Soul]
14        15   Electronica/Dance  [Electronica, Dance]
15        16               World               [World]
16        17         Hip Hop/Rap        [Hip Hop, Rap]
18        

In [15]:
# Collect all unique genre names from the original table.
genre_set = set()

for row in df_original_db_genre_table["name_split"]:
    genre_set.update(row)
    
# Convert the set to a sorted list, so we can iterate in a stable order.
genre_list = sorted(genre_set)

In [16]:
# Create an empty DataFrame for the normalized "genre" table.
df_normalized_db_genre_table = pd.DataFrame(columns=["genre_id", "genre_name"])

# Use the elements inside "genre_list" to fill the "df_normalized_db_genre_table" DataFrame.
for index, genre in enumerate(genre_list, start=1):
    df_normalized_db_genre_table.loc[len(df_normalized_db_genre_table)] = {
        "genre_id": index,
        "genre_name": genre
    }

# Set the datatypes.
df_normalized_db_genre_table = df_normalized_db_genre_table.astype({
    "genre_id": "Int64",
    "genre_name": "string"
})

print(df_normalized_db_genre_table)

    genre_id      genre_name
0          1     Alternative
1          2           Blues
2          3      Bossa Nova
3          4       Classical
4          5          Comedy
5          6           Dance
6          7           Drama
7          8  Easy Listening
8          9     Electronica
9         10         Fantasy
10        11     Heavy Metal
11        12         Hip Hop
12        13            Jazz
13        14           Latin
14        15           Metal
15        16           Opera
16        17             Pop
17        18            Punk
18        19             R&B
19        20             Rap
20        21          Reggae
21        22            Rock
22        23   Rock And Roll
23        24          Sci Fi
24        25            Soul
25        26      Soundtrack
26        27        TV Shows
27        28           World


In [17]:
# In order to simplify the creation of the junction DataFrame "genre_track", we enrich
# the "df_original_db_genre_table" DataFrame adding a new column that contains the
# new IDs for each music genre.

# Firstly create a dictionary from the "df_normalized_db_genre_table" DataFrame.
df_dict = dict(zip(df_normalized_db_genre_table["genre_name"], df_normalized_db_genre_table["genre_id"]))

In [18]:
# Create a new column called "new_genre_id" and fill each row with an empty list.
df_original_db_genre_table["new_genre_id"] = [[] for _ in range(len(df_original_db_genre_table))]

# Iterate over the genres and when the same genre is found, assign the corresponding
# new genre ID to the "new_genre_id" column of the "df_original_db_genre_table" DataFrame.
for index, row in df_original_db_genre_table.iterrows():
    for element in row["name_split"]:
        if element in df_dict:
            value = df_dict[element]
            df_original_db_genre_table.at[index, "new_genre_id"].append(value)

print(df_original_db_genre_table)
df_original_db_genre_table.info()

    genre_id                name            name_split new_genre_id
0          1                Rock                [Rock]         [22]
1          2                Jazz                [Jazz]         [13]
2          3               Metal               [Metal]         [15]
3          4  Alternative & Punk   [Alternative, Punk]      [1, 18]
4          5       Rock And Roll       [Rock And Roll]         [23]
5          6               Blues               [Blues]          [2]
6          7               Latin               [Latin]         [14]
7          8              Reggae              [Reggae]         [21]
8          9                 Pop                 [Pop]         [17]
9         10          Soundtrack          [Soundtrack]         [26]
10        11          Bossa Nova          [Bossa Nova]          [3]
11        12      Easy Listening      [Easy Listening]          [8]
12        13         Heavy Metal         [Heavy Metal]         [11]
13        14            R&B/Soul           [R&B,

In [19]:
# Create a dictionary that contains key-value pairs of the old and new genre IDs.
df_genre_id_dict = dict(zip(df_original_db_genre_table["genre_id"], df_original_db_genre_table["new_genre_id"]))

In [20]:
rows = []

# Using the previously created "df_original_db_track_table" DataFrame and the
# "df_genre_id_dict" dictionary, store the "genre_id" and "track_id" in the "rows" list.
for track in df_original_db_track_table.itertuples():
    old_id = track.genre_id
    new_ids = df_genre_id_dict[old_id]
    
    for new_id in new_ids:
        rows.append({
            "genre_id": new_id,
            "track_id": track.track_id
        })

# Create the Dataframe.
df_normalized_db_genre_track_table = pd.DataFrame(rows)
df_normalized_db_genre_track_table["genre_track_id"] = range(1, len(rows)+1)

# Set the datatypes.
df_normalized_db_genre_track_table = df_normalized_db_genre_track_table.astype({
    "genre_track_id": "Int64",
    "genre_id": "Int64",
    "track_id": "Int64",
})

# Reorder the columns, so that the DataFrame matches the "genre_track" junction table of the normalized database.
df_normalized_db_genre_track_table = df_normalized_db_genre_track_table[ ["genre_track_id", "genre_id", "track_id"] ]

df_normalized_db_genre_track_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 4040 entries, 0 to 4039
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   genre_track_id  4040 non-null   Int64
 1   genre_id        4040 non-null   Int64
 2   track_id        4040 non-null   Int64
dtypes: Int64(3)
memory usage: 106.7 KB


In [21]:
# Create the "media_type" DataFrame.
df_normalized_db_media_type_table = pd.read_sql("SELECT * FROM media_type", engine_original_db)

# Rename the columns.
df_normalized_db_media_type_table = df_normalized_db_media_type_table.rename(
    columns={"name": "media_type_name"}
)

# Define the column datatypes.
schema = {
    "media_type_id": "Int64",
    "media_type_name": "string"
}

# Apply the datatypes.
df_normalized_db_media_type_table = df_normalized_db_media_type_table.astype(schema)

print(df_normalized_db_media_type_table)
df_normalized_db_media_type_table.info()

   media_type_id              media_type_name
0              1              MPEG audio file
1              2     Protected AAC audio file
2              3  Protected MPEG-4 video file
3              4     Purchased AAC audio file
4              5               AAC audio file
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   media_type_id    5 non-null      Int64 
 1   media_type_name  5 non-null      string
dtypes: Int64(1), string(1)
memory usage: 321.0 bytes


In [22]:
# Create a Dataframe that contains the "track_id" and "composer" columns from the "df_original_db_track_table" df.
# This will be used to normalize and split the composer strings.
df_track_id_unormalized_composer = pd.DataFrame().assign(track_id=df_original_db_track_table["track_id"], unormalized_composer=df_original_db_track_table["composer"])
df_track_id_unormalized_composer.head()

,track_id,unormalized_composer
0,1,"Angus Young, Malcolm Young, Brian Johnson"
1,2,<NA>
2,3,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho..."
3,4,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D..."
4,5,Deaffy & R.A. Smith-Diesel


In [23]:
# Create a function that will handle:
# - Unicode character differences
# - Different hyphens
# - Whitespaces
def canonicalize_manual_string(s):
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("–", "-").replace("—", "-")
    s = " ".join(s.split())
    # Return the string with lowercase letters, so that case sensitivity is not an issue during manual map lookup.
    return s.lower()

In [24]:
# The following dictionary defines explicit splits for 
# raw composer strings that are too irregular or inconsistent
# to be handled by normalization using regex or separator rules.

# These entries represent exception cases, where the original database format uses:
# - Non‑ASCII hyphens or Unicode punctuation
# - Surname‑first formats
# - Band/member shorthand
# - Character encoding errors
# - Inconsistent separators 
# - Multi‑name compounds that cannot be split using a general pattern

composer_manual_split_dict = {
    "A.Bouchard/J.Bouchard/S.Pearlman": ["A. Bouchard", "J. Bouchard", "S. Pearlman"],
    "A.Isbell/A.Jones/O.Redding": ["A. Isbell", "A. Jones", "O. Redding"],
    "AC/DC": ["AC/DC"],
    "Acyi Marques/Arlindo Bruz/Braço, Beto Sem/Zeca Pagodinho": ["Acyi Marques", "Arlindo Bruz", "Beto Sem Braço", "Zeca Pagodinho"],
    "Anonymous": [],
    "Arlindo Cruz/Franco/Marquinhos PQD/Negro, Jovelina Pérolo/Zeca Pagodinho": ["Arlindo Cruz", "Franco", "Marquinhos PQD", "Jovelina Pérolo Negro", "Zeca Pagodinho"],
    "Arnaldo Antunes/Britto, SergioMiklos, Paulo": ["Arnaldo Antunes", "Sergio Britto", "Paulo Miklos"],
    "Arnaldo Baptista - Rita Lee": ["Arnaldo Baptista", "Rita Lee"],
    "Arnaldo Baptista - Rita Lee -  Sérgio Dias": ["Arnaldo Baptista", "Rita Lee", "Sérgio Dias"],
    "Arnaldo Baptista - Rita Lee - Arnolpho Lima Filho": ["Arnaldo Baptista", "Rita Lee", "Arnolpho Lima Filho"],
    "Arnaldo Baptista - Rita Lee - Sérgio Dias": ["Arnaldo Baptista", "Rita Lee", "Sérgio Dias"],
    "Augusto/Craig Negoescu/Rocco Filho/Saara, Ze Carlos Da": ["Augusto", "Craig Negoescu", "Rocco Filho", "Ze Carlos Da Saara"],
    "B. Cummings/G. Peterson/M.J. Kale/R. Bachman": ["B. Cummings", "G. Peterson", "M. J. Kale", "R. Bachman"],
    "B. de Filippi - F. Migliaci - Versão: Fred Jorge": ["B. de Filippi", "F. Migliaci", "Fred Jorge"],
    "Ben E.King/James Bethea": ["Ben E. King", "James Bethea"],
    "Berry Gordy, Jr./Janie Bradford": ["Berry Gordy Jr", "Janie Bradford"],
    "Beto Guedes/Fernando Brant/L Borges": ["Beto Guedes", "Fernando Brant", "L. Borges"],
    "Bill Berry-Peter Buck-Mike Mills-Michael Stipe": ["Bill Berry", "Peter Buck", "Mike Mills", "Michael Stipe"],
    "Billie Joe Armstrong -Words Green Day -Music": ["Billie Joe Armstrong", "Green Day"],
    "Blackmore, Gillan, Glover, Lord, Paice": ["Ritchie Blackmore", "Ian Gillan", "Roger Glover", "Jon Lord", "Ian Paice"],
    "Blackmore, Glover, Turner": ["Ritchie Blackmore", "Roger Glover", "Joe Lynn Turner"],
    "Blackmore, Glover, Turner, Lord, Paice": ["Ritchie Blackmore", "Roger Glover", "Joe Lynn Turner", "Jon Lord", "Ian Paice"],
    "Blackmore/Coverdale": ["Ritchie Blackmore", "David Coverdale"],
    "Blackmore/Coverdale/Lord/Paice": ["Ritchie Blackmore", "David Coverdale", "Jon Lord", "Ian Paice"],
    "Blackmore/Gillan/Glover/Lord/Paice": ["Ritchie Blackmore", "Ian Gillan", "Roger Glover", "Jon Lord", "Ian Paice"], 
    "Bono/Clayton, Adam/Dylan, Bob/Mullen Jr., Larry/The Edge": ["Bono", "Adam Clayton", "Bob Dylan", "Larry Mullen Jr", "The Edge"],
    "Billy Roberts": ["Billy Roberts"],
    "Bono/Clayton, Adam/Mullen Jr., Larry/The Edge": ["Bono", "Adam Clayton", "Larry Mullen Jr", "The Edge"],
    "Bolin/Coverdale": ["Tommy Bolin", "David Coverdale"],
    "Bolin/Coverdale/Paice": ["Tommy Bolin", "David Coverdale", "Ian Paice"],
    "Bolin/Hughes": ["Tommy Bolin", "Glenn Hughes"],
    "Bolin/Hughes/Lord": ["Tommy Bolin", "Glenn Hughes", "Jon Lord"],
    "Bossi/David Coverdale/Earl Slick": ["Doug Bossi", "David Coverdale", "Earl Slick"],
    "Brian Eno, Bono, Adam Clayton, The Edge & Larry Mullen Jnr.": ["Brian Eno", "Bono", "Adam Clayton", "The Edge", "Larry Mullen Jr"],
    "Bruce/Clapton":  ["Jack Bruce", "Eric Clapton"],
    "Bruce Dickinson; Harris": ["Bruce Dickinson", "Steve Harris"],
    "Bourge/Phillips/Shelley": ["Tony Bourge", "Ray Phillips", "Burke Shelley"],
    "Buchanan, Wallis": ["Wallis Buchanan"],
    "C. A./C.A./Celso Alvim/Herbert Vianna/Mário Moura/Pedro Luís/Sidon Silva": ["Celso Alvim", "Herbert Vianna", "Mário Moura", "Pedro Luís", "Sidon Silva"],
    "C.Burton": ["C. Burton"],
    "Caetano Veloso - Djavan": ["Caetano Veloso", "Djavan"],
    "Caetano Veloso - Gilberto Gil": ["Caetano Veloso", "Gilberto Gil"],
    "Caetano Veloso - Waly Salomão": ["Caetano Veloso", "Waly Salomão"],
    "Caetano Veloso e Gilberto Gil": ["Caetano Veloso", "Gilberto Gil"],
    "Cal Adan/Del Rey, Tension/Edu Casanova": ["Cal Adan", "Tension Del Rey", "Edu Casanova"],
    "Carlos Santana & KC Porter": ["Carlos Santana", "K. C. Porter"],
    "Cale/Clapton": ["J. J. Cale", "Eric Clapton"],
    "Cassiano Costa/Cintia Maviane/J.F./Lucas Costa": ["Cassiano Costa", "Cintia Maviane", "J. F.", "Lucas Costa"],
    "Clapton": ["Eric Clapton"],
    "Clapton/Collins/Pappalardi": ["Eric Clapton", "Gail Collins", "Felix Pappalardi"],
    "Clapton/Dylan": ["Eric Clapton", "Bob Dylan"],
    "Clapton/F.eldman/Linn": ["Eric Clapton", "Richard Feldman", "Roger Linn"],
    "Clapton/Gordon": ["Eric Clapton", "Jim Gordon"],
    "Clapton/Harrison": ["Eric Clapton", "George Harrison"],
    "Clapton/J. J. Cale": ["Eric Clapton", "J. J. Cale"],
    "Clapton/Levy": ["Eric Clapton", "Marcy Levy"],
    "Clapton/Robert Johnson Arr: Eric Clapton": ["Robert Johnson", "Eric Clapton"],
    "Clapton/Trad. Arr. Clapton": ["Eric Clapton"],
    "Clarke/Farren/Kilmister/Tayler": ["Eddie Clarke", "Mick Farren", "Lemmy Kilmister", "Phil Taylor"],
    "Clarke/Kilmister/Tayler": ["Eddie Clarke", "Lemmy Kilmister", "Phil Taylor"],
    "Clarke/Kilmister/Taylor": ["Eddie Clarke", "Lemmy Kilmister", "Phil Taylor"],
    "Claudio Martins/David Lima/Kleber Rodrigues/Livre, Cesare Som": ["Claudio Martins", "David Lima", "Kleber Rodrigues", "Cesare Som Livre"],
    "Cook/Coverdale": ["Jeff Cook", "David Coverdale"],
    "Collins/Van Zandt": ["Allen Collins", "Ronnie Van Zant"],
    "Cornell, Commerford, Morello, Wilk": ["Chris Cornell", "Tim Commerford", "Tom Morello", "Brad Wilk"],
    "Coverdale": ["David Coverdale"],
    "Coverdale/Hughes": ["David Coverdale", "Glenn Hughes"],
    "Coverdale/Hughes/Lord/Paice": ["David Coverdale", "Glenn Hughes", "Jon Lord", "Ian Paice"],
    "Coverdale/Lord/Paice": ["David Coverdale", "Jon Lord", "Ian Paice"],
    "Culmer/Exalt": ["Nick Culmer", "Chris Exall"],
    "D.Coverdal/G.Hughes/Glenn Hughes/I.Paice/Ian Paice/J.Lord/John Lord": ["David Coverdale", "Glenn Hughes", "Ian Paice", "Jon Lord"],
    "D.Coverdale/G.Hughes/Glenn Hughes/I.Paice/Ian Paice/J.Lord/John Lord/R.Blackmore/Ritchie Blackmore": ["David Coverdale", "Glenn Hughes", "Ian Paice", "Jon Lord", "Ritchie Blackmore"],
    "D.Coverdale/G.Hughes/Glenn Hughes/J.Lord/John Lord": ["David Coverdale", "Glenn Hughes", "Ian Paice", "Jon Lord"],
    "D.Coverdale/G.Hughes/Glenn Hughes/R.Blackmore/Ritchie Blackmore": ["David Coverdale", "Glenn Hughes", "Ritchie Blackmore"],
    "D.Coverdale/R.Blackmore/Ritchie Blackmore": ["David Coverdale", "Ritchie Blackmore"],
    "D. DeLeo/Kretz/R. DeLeo/Weiland": ["Dean DeLeo", "Eric Kretz", "Robert DeLeo", "Scott Weiland"],
    "D. DeLeo/Kretz/Weiland": ["Dean DeLeo", "Eric Kretz", "Scott Weiland"],
    "D. DeLeo/R. DeLeo/Weiland": ["Dean DeLeo", "Robert DeLeo", "Scott Weiland"],
    "Darius Take One Minwalla/Jon Auer/Ken Stringfellow/Matt Harris": ["Darius 'Take One' Minwalla", "Jon Auer", "Ken Stringfellow", "Matt Harris"],
    "Dave Kushner, Duff, Keith Nelson, Matt Sorum, Scott Weiland & Slash": ["Dave Kushner", "Duff McKagan", "Keith Nelson", "Matt Sorum", "Scott Weiland", "Slash"],
    "Dave Kushner, Duff, Matt Sorum, Scott Weiland & Slash": ["Dave Kushner", "Duff McKagan", "Matt Sorum", "Scott Weiland", "Slash"],
    "Dave, Taylor, Nate, Chris": ["Dave Grohl", "Taylor Hawkins", "Nate Mendel", "Chris Shiflett"],
    "Dadinho/Gilbreto Gomes/Gustavo/P.C. Portugal/R. Mocoto": ["Dadinho", "Gilbreto Gomes", "Gustavo", "P. C. Portugal", "R. Mocoto"],
    "David Corrêa - Paulinho Carvalho - Carlos Sena - Bira do Ponto": ["David Corrêa", "Paulinho Carvalho", "Carlos Sena", "Bira do Ponto"],
    "De Beauport/Jagger/Richards": ["Pierre de Beauport", "Mick Jagger", "Keith Richards"],
    "Deacon": ["John Deacon"],
    "Deacon, John": ["John Deacon"],
    "Deacon, John/May, Brian": ["John Deacon", "Brian May"],
    "Deacon/May/Mercury/Taylor": ["John Deacon", "Brian May", "Freddie Mercury", "Roger Taylor"],
    "Deaffy & R.A. Smith-Diesel": ["Deaffy", "R. A. Smith-Diesel"],
    "Di´Anno/Harris": ["Paul Di'Anno", "Steve Harris"],
    "Diamond/Shermann": ["King Diamond", "Hank Shermann"],
    "Dickinson": ["Bruce Dickinson"],
    "Dickinson/Smith": ["Bruce Dickinson","Adrian Smith"],
    "Dito/Djalma Falcao/Ilha, Almir Da/Márcio André": ["Dito", "Djalma Falcao", "Almir Da Ilha", "Márcio André"],
    "Djavan - Gabriel, O Pensador": ["Djavan", "Gabriel o Pensador"],
    "Do Vale, João/Luiz Wanderley": ["João Do Vale", "Luiz Wanderley"],
    "Do Vale, Valverde/Gal Oliveira/Luciano Pinto": ["Valverde Do Vale", "Gal Oliveira", "Luciano Pinto"],
    "Dolmayan, John/Malakian, Daron/Odadjian, Shavo": ["John Dolmayan", "Daron Malakian", "Shavo Odadjian"],
    "Domenil/J. Brito/Joaozinho/Rap, Marcelo Do": ["Domenil", "J. Brito", "Joaozinho", "Marcelo Do Rap"],
    "Douglas/Neves, Vicente Das/Silva, Gilmar L./Toninho Gentil/Wantuir": ["Douglas", "Vicente Das Neves", "Gilmar L. Silva", "Toninho Gentil", "Wantuir"],
    "Dylan, Bob": ["Bob Dylan"],
    "Dufort/Johnson/McAuliffe/Williams": ["Denise Dufort", "Kelly Johnson", "Kim McAuliffe", "Enid Williams"],
    "E.Weiss": ["E. Weiss"],
    "Eale-Eye Cherry, M. Simpson, J. King & M. Nishita": ["Eagle-Eye Cherry", "M. Simpson", "J. King", "M. Nishita"],
    "Edward Van Halen, Alex Van Halen, Michael Anthony,/Edward Van Halen, Alex Van Halen, Michael Anthony, David Lee Roth": ["Edward Van Halen", "Alex Van Halen", "Michael Anthony", "David Lee Roth"],
    "Edward Van Halen, Alex Van Halen, Michael Anthony,/Edward Van Halen, Alex Van Halen, Michael Anthony, Sammy Hagar": ["Edward Van Halen", "Alex Van Halen", "Michael Anthony", "Sammy Hagar"],
    "F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. Dirkscneider & W. Hoffman": ["F. Baltes", "R. A. Smith-Diesel", "S. Kaufman", "U. Dirkscneider", "W. Hoffman"],
    "Faithfull/Jagger/Richards": ["Marianne Faithfull", "Mick Jagger", "Keith Richards"],
    "Fausto Nilo - Armandinho": ["Fausto Nilo", "Armandinho"],
    "G Harvey/R Hope-Taylor": ["G. Harvey", "R. Hope-Taylor"],
    "G M Sumner": ["G. M. Sumner"],
    "Garay, Ricardo Engels/Luca Predabom/Ludwig, Carlos Henrique/Maurício Vieira": ["Ricardo Engels Garay", "Luca Predabom", "Carlos Henrique Ludwig", "Maurício Vieira"],
    "Gelder, Nick van": ["Nick van Gelder"],
    "Gilberto Gil - Caetano Veloso": ["Gilberto Gil", "Caetano Veloso"],
    "Gillan/Glover/Lord/Nix - Blackmore/Paice": ["Ian Gillan", "Roger Glover", "Jon Lord", "Ritchie Blackmore", "Ian Paice"],
    "Gillan/Glover/Lord/Paice": ["Ian Gillan", "Roger Glover", "Jon Lord", "Ian Paice"],
    "Gilmour, Mason, Wright, Waters": ["David Gilmour", "Nick Mason", "Richard Wright", "Roger Waters"],
    "Gilmour, Waters": ["David Gilmour", "Roger Waters"],
    "Gonzaga Jr.": ["Gonzaga Jr"],
    "Guio De Morais E Seus Parentes/Luiz Gonzaga": ["Guio De Morais E Seus Parentes", "Luiz Gonzaga"],
    "Harris": ["Steve Harris"],
    "Harris/Di Anno": ["Steve Harris", "Paul Di'Anno"],
    "Harris/Paul Di´Anno": ["Steve Harris", "Paul Di'Anno"],
    "Harris/Tatler": ["Sean Harris", "Brian Tatler"],
    "Hawkins-Lewis-Broadwater": ["Dale Hawkins", "Stan Lewis", "Eleanor Broadwater"],
    "Heath/Robinson": ["Fred Heath", "Guy Robinson"],
    "Hetfield, Ulrich": ["James Hetfield", "Lars Ulrich"],
    "Hetfield, Ulrich, Hammett": ["James Hetfield", "Lars Ulrich", "Kirk Hammett"],
    "Hetfield, Ulrich, Newsted": ["James Hetfield", "Lars Ulrich", "Jason Newsted"], 
    "Hendrix, Jimi": ["Jimi Hendrix"],
    "Henry Kirssch/Lenny Kravitz/S. Pasch A. Krizan": ["Henry Kirssch", "Lenny Kravitz", "S. Pasch", "A. Krizan"],
    "Ian Gillan/Ian Paice/Jon Lord/Ritchie Blckmore/Roger Glover": ["Ian Gillan", "Ian Paice", "Jon Lord", "Ritchie Blackmore", "Roger Glover"],
    "J B Lenoir/Willie Dixon": ["J. B. Lenoir", "Willie Dixon"],
    "J. Kay/Jay Kay": ["Jay Kay"],
    "J.C. Fogerty": ["J. C. Fogerty"],
    "J.Hetfield/L.Ulrich": ["James Hetfield", "Lars Ulrich"],
    "J.J. Johnson": ["J. J. Johnson"],
    "J.J. Johnson/R. Troup, L. Worth": ["J. J. Johnson", "R. Troup", "L. Worth"],
    "J.r.Bertami/Parana": ["J. R. Bertami", "Parana"],
    "Jagger/Richards": ["Mick Jagger", "Keith Richards"],
    "Jagger/Richards/Oldham": ["Mick Jagger", "Keith Richards", "Andrew Loog Oldham"],
    "Jay Kay/Kay, Jay": ["Jay Kay"],
    "Jimmy and Vella Cameron": ["Jimmy Cameron", "Vella Cameron"],
    "John and William Valentine": ["John Valentine", "William Valentine"],
    "Jones/Sirotto/Smith": ["Ian Jones", "Jim Sirotto", "Brian Smith"],
    "K.Hammett": ["Kirk Hammett"],
    "KC Porter & JB Eckl": ["K. C. Porter", "J. B. Eckl"],
    "Kelly, Mosley, Rothery, Trewaves": ["Mark Kelly", "Ian Mosley", "Steve Rothery", "Pete Trewavas"],
    "Kretz/R. DeLeo/Weiland": ["Eric Kretz", "Robert DeLeo", "Scott Weiland"],
    "Leão, Rodrigo F./Lô Borges/Samuel Rosa": ["Rodrigo F. Leão", "Lô Borges", "Samuel Rosa"],
    "Leão, Rodrigo F./Samuel Rosa": ["Rodrigo F. Leão", "Samuel Rosa"],
    "Lennon, John/McCartney, Paul": ["John Lennon", "Paul McCartney"],
    "M.Babatunde Olantunji": ["M. Babatunde Olantunji"],
    "Mabins, Macie/Magee, Sterling/Robinson, Bobby": ["Macie Mabins", "Sterling Magee", "Bobby Robinson"],
    "Marisa Monte/poema de Octavio Paz/versão: Haroldo de Campos": ["Marisa Monte", "Octavio Paz", "Haroldo de Campos"],
    "Maxton Gig Beesley Jnr.": ["Maxton Gig Beesley Jr"],
    "Marley": ["Bob Marley"],
    "Marsden": ["Bernie Marsden"],
    "Marsden/Moody": ["Bernie Marsden", "Micky Moody"],
    "May": ["Brian May"],
    "May, Brian": ["Brian May"],
    "Mason, Waters, Wright, Gilmour": ["Nick Mason", "Roger Waters", "Richard Wright", "David Gilmour"],
    "Mason/Waters, Gilmour, Wright": ["Nick Mason", "Roger Waters", "Richard Wright", "David Gilmour"],
    "Mercury": ["Freddie Mercury"],
    "Mercury, Freddie": ["Freddie Mercury"],
    "Mike Dirnt -Words Green Day -Music": ["Mike Dirnt", "Green Day"],
    "Molaney/Morris/Roberts/Wainwright": ["Garry Maloney", "Kelvin Morris", "Tony Roberts", "Roy Wainwright"],
    "Mutantes": ["Os Mutantes"],
    "Orestes Barbosa-Sílvio Caldas": ["Orestes Barbosa", "Sílvio Caldas"],
    "Os Mutantes - Arnolpho Lima Filho": ["Os Mutantes", "Arnolpho Lima Filho"],
    "Page, Jones": ["Jimmy Page", "John Paul Jones"],
    "R. DeLeo/Weiland": ["Robert DeLeo", "Scott Weiland"],
    "R.E.M": ["R.E.M."],
    "Renato Russo - Marcelo Bonfá": ["Renato Russo", "Marcelo Bonfá"],
    "Renato Russo - Marisa Monte": ["Renato Russo", "Marisa Monte"],
    "Rita Lee - Arnaldo Baptista - Sérgio Baptista": ["Rita Lee", "Arnaldo Baptista", "Sérgio Baptista"],
    "Richie Blackmore, Ian Gillian, Roger Glover": ["Ritchie Blackmore", "Ian Gillan", "Roger Glover"],
    "Richie Blackmore, Ian Gillian, Roger Glover, Ian Paice": ["Ritchie Blackmore", "Ian Gillan", "Roger Glover", "Ian Paice"],
    "Richie Blackmore, Ian Gillian, Roger Glover, Jon Lord": ["Ritchie Blackmore", "Ian Gillan", "Roger Glover", "Jon Lord"],
    "Richie Blackmore, Ian Gillian, Roger Glover, Jon Lord, Ian Paice": ["Ritchie Blackmore", "Ian Gillan", "Roger Glover", "Jon Lord", "Ian Paice"],
    "Santana": ["Carlos Santana"],
    "Seger": ["Bob Seger"],
    "Smith, Toby": ["Toby Smith"],
    "Smith/Dickinson": ["Adrian Smith", "Bruce Dickinson"],
    "Tankian, Serj": ["Serj Tankian"],
    "Taylor": ["Roger Taylor"],
    "Tom Jobim - Newton Mendoça": ["Tom Jobim", "Newton Mendoça"],
    "Turner, Held, Greenwood": ["Joe Lynn Turner", "Bob Held", "Al Greenwood"],
    "U2 & Daragh O'Toole": ["U2", "Darragh O'Toole"],
    "U2; Bono & Edge, The": ["U2", "Bono", "The Edge"],
    "U2; Edge, The": ["U2", "The Edge"],
    "Ulrich": ["Lars Ulrich"],
    "W.Correa": ["W. Correa"],
    "Waters": ["Roger Waters"],
    "Waters, Wright": ["Roger Waters", "Richard Wright"],
    "Weill-Brecht": ["Kurt Weill", "Bertolt Brecht"],
    "Whitfield-Strong": ["Norman Whitfield", "Barrett Strong"],
    "Wright, Waters": ["Richard Wright", "Roger Waters"],
}

In [25]:
# Create a new dictionary with canonicalized keys.
canonicalized_composer_manual_split_dict = {canonicalize_manual_string(raw_composer_string): composer_manual_split_dict[raw_composer_string] for raw_composer_string in composer_manual_split_dict}

In [26]:
# To split the raw composer or artist strings not included in the mapping dictionaries,
# we need to create a function that handles the normalization logic for separators.

# Create a function that converts punctuation and conjunctions into a single "||" separator.
def normalize_separators(s):
    if s is None or pd.isna(s):
        return ""

    s = str(s)

    # Normalize Unicode hyphens/dashes to ASCII hyphen.
    s = re.sub(r"[‐‑‒–—−]", "-", s)

    # Protect sequences of initials like "A. B. C.".
    s = re.sub(r"\b([A-Z]\.)\s+(?=[A-Z]\.)", r"\1", s)

    # Normalize slashes as separators.
    s = re.sub(r"\s*/\s*", "||", s)

    # Normalize commas as separators.
    s = re.sub(r"\s*,\s*", "||", s)

    # Normalize semicolons as separators.
    s = re.sub(r"\s*;\s*", "||", s)

    # Normalize conjunctions (whole-word "and"), anywhere in the string.
    s = re.sub(r"\band\b", "||", s, flags=re.IGNORECASE)

    # Normalize ampersands.
    s = re.sub(r"\s*&\s*", "||", s)

    # Convert various role/credit tokens into the selected separator.
    s = re.sub(r"\bTraditional\s+arr\.?\s*", "||", s, flags=re.IGNORECASE)
    s = re.sub(r"\bArr\.?:\b", "||", s, flags=re.IGNORECASE)
    s = re.sub(r"\bTrad\.?\b", "||", s, flags=re.IGNORECASE)
    s = re.sub(r"-Music\b", "||", s, flags=re.IGNORECASE)
    s = re.sub(r"-Words\b", "||", s, flags=re.IGNORECASE)

    # Collapse multiple separators.
    s = re.sub(r"\|\|+", "||", s)

    # Trim leading/trailing separators.
    s = s.strip("|")

    return s

In [27]:
# Create a function that will handle the normalization of each composer or artist name, after the strings are split.

# The function performs the following normalizations:
# - trim whitespace
# - normalize spacing around periods
# - collapse multiple spaces
def normalize_atomic_name(s):
    
    if not s:
        return ""

    # Trim leading/trailing whitespace.
    s = s.strip()

    # Add exceptions for specific composers and artists.
    special_cases_dict = {
        "ac/dc": "AC/DC",
        "dado villa-lobos": "Dado Villa-Lobos",
        "darius 'take one' minwalla": "Darius 'Take One' Minwalla",
        "darragh o'toole": "Darragh O'Toole",
        "derrick mckenzie": "Derrick McKenzie",
        "duff mckagan": "Duff McKagan",
        "eagle-eye cherry": "Eagle-Eye Cherry",
        "fritz mcintyre": "Fritz McIntyre",
        "gabriel o pensador": "Gabriel O Pensador",
        "ico ouro-preto": "Ico Ouro-Preto",
        "jesse edwin davis iii": "Jesse Edwin Davis III",
        "kim mcauliffe": "Kim McAuliffe",
        "mckinley morgenfield (muddy waters)": "McKinley Morgenfield (Muddy Waters)",
        "marc-antoine charpentier": "Marc-Antoine Charpentier",
        "marquinhos pqd": "Marquinhos PQD",
        "marelo d'aguia": "Marelo D'Aguia",
        "mike mccready": "Mike McCready",
        "nicko mcbrain": "Nicko McBrain",
        "nikolai rimsky-korsakov": "Nikolai Rimsky-Korsakov",
        "paul di'anno": "Paul Di'Anno",
        "paul mccartney": "Paul McCartney",
        "paul o'duffy": "Paul O'Duffy",
        "pierre de beauport": "Pierre de Beauport",
        "raimundinho doacordion": "Raimundinho DoAcordion",
        "r. a. smith-diesel": "R. A. Smith-Diesel",
        "r. hope-taylor": "R. Hope-Taylor",
        "rod mckuen": "Rod McKuen",
        "winston foster-vers": "Winston Foster-Vers",
        "winston foser-vers": "Winston Foser-Vers",
    }
    
    lower_s = s.lower()
    if lower_s in special_cases_dict:
        return special_cases_dict[lower_s]

    # Normalize spacing around periods to handle initials formatting.
    s = re.sub(r"\.(?!\s)", ". ", s)
    s = re.sub(r"\s+\.", ".", s)

    # Collapse multiple spaces.
    s = re.sub(r"\s+", " ", s).strip()

    # Capitalize each word.
    s = " ".join(word.capitalize() for word in s.split())
    
    return s

In [28]:
# Create a function that will handle the raw composer strings whether they appear on the mapping dictionary or not.
def process_composer_string(s):
    # Handle missing values
    if s is None or pd.isna(s):
        return None

    canonical = canonicalize_manual_string(s)
    
    # If the string is in the manual mapping dictionary, return a message.
    if canonical in canonicalized_composer_manual_split_dict:
        return "Composer string handled by the mapping dictionary."
    # Otherwise, normalize the separators and return the normalized string.
    return normalize_separators(s)
    
# Apply the above function and create a new column in the DataFrame.
df_track_id_unormalized_composer["normalized_separators"] = (
    df_track_id_unormalized_composer["unormalized_composer"]
    .apply(process_composer_string)
)

In [29]:
df_track_id_unormalized_composer.head()

,track_id,unormalized_composer,normalized_separators
0,1,"Angus Young, Malcolm Young, Brian Johnson",Angus Young||Malcolm Young||Brian Johnson
1,2,<NA>,NaN
2,3,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",F. Baltes||S. Kaufman||U. Dirkscneider||W. Hof...
3,4,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",Composer string handled by the mapping diction...
4,5,Deaffy & R.A. Smith-Diesel,Composer string handled by the mapping diction...


In [30]:
# Add a new column to the DataFrame, that contains a list with the composers of each track per row.
def track_composer_list(s):
    # Handle missing values
    if s is None or pd.isna(s):
        return None

    # Canonicalize the input.
    canonical = canonicalize_manual_string(s)

    # Case 1: handled manually.
    if canonical in canonicalized_composer_manual_split_dict:
        return canonicalized_composer_manual_split_dict[canonical]

    # Case 2: normalize & split.
    normalized = normalize_separators(s)
    return normalized.split("||")

df_track_id_unormalized_composer["track_composer_list"] = (
    df_track_id_unormalized_composer["unormalized_composer"]
    .apply(track_composer_list)
)

# Remove duplicate composers from each row's "track_composer_list" 
# while preserving the original order, if the value is a list.
df_track_id_unormalized_composer["track_composer_list"] = (
    df_track_id_unormalized_composer["track_composer_list"]
    .apply(lambda x: list(dict.fromkeys(x)) if isinstance(x, list) else x)
)

In [31]:
df_track_id_unormalized_composer.head()

,track_id,unormalized_composer,normalized_separators,track_composer_list
0,1,"Angus Young, Malcolm Young, Brian Johnson",Angus Young||Malcolm Young||Brian Johnson,"[Angus Young, Malcolm Young, Brian Johnson]"
1,2,<NA>,NaN,None
2,3,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",F. Baltes||S. Kaufman||U. Dirkscneider||W. Hof...,"[F. Baltes, S. Kaufman, U. Dirkscneider, W. Ho..."
3,4,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",Composer string handled by the mapping diction...,"[F. Baltes, R. A. Smith-Diesel, S. Kaufman, U...."
4,5,Deaffy & R.A. Smith-Diesel,Composer string handled by the mapping diction...,"[Deaffy, R. A. Smith-Diesel]"


In [32]:
# Create a set of words that are not valid composers/bands that appear in the data, in order to exclude them.
excluded_words = {
    "traditional",
    "tradicional",
    "traditionnel",
    "folclore" # 'folklore' typo of Folklore
    "varios",
    "vários",
    "various",
    "diversos",
    "several",
    "multiple",
}

# Create a function that will normalize each atomic composer name using the previously created "normalize_atomic_name" function.
def normalize_composer_list(lst):
    # Handle missing values
    if lst is None or lst is pd.NA:
        return None
    
    cleaned = []
    
    for name in lst:
        atomic = normalize_atomic_name(name)
        
        # Skip unwanted role words
        if atomic.lower() in excluded_words:
            continue
        
        cleaned.append(atomic)
    
    return cleaned

df_track_id_unormalized_composer["normalized_composer_list"] = (
    df_track_id_unormalized_composer["track_composer_list"]
    .apply(normalize_composer_list)
)

In [33]:
df_track_id_unormalized_composer.head()

,track_id,unormalized_composer,normalized_separators,track_composer_list,normalized_composer_list
0,1,"Angus Young, Malcolm Young, Brian Johnson",Angus Young||Malcolm Young||Brian Johnson,"[Angus Young, Malcolm Young, Brian Johnson]","[Angus Young, Malcolm Young, Brian Johnson]"
1,2,<NA>,NaN,None,None
2,3,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",F. Baltes||S. Kaufman||U. Dirkscneider||W. Hof...,"[F. Baltes, S. Kaufman, U. Dirkscneider, W. Ho...","[F. Baltes, S. Kaufman, U. Dirkscneider, W. Ho..."
3,4,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",Composer string handled by the mapping diction...,"[F. Baltes, R. A. Smith-Diesel, S. Kaufman, U....","[F. Baltes, R. A. Smith-Diesel, S. Kaufman, U...."
4,5,Deaffy & R.A. Smith-Diesel,Composer string handled by the mapping diction...,"[Deaffy, R. A. Smith-Diesel]","[Deaffy, R. A. Smith-Diesel]"


In [34]:
# Create a set to collect unique composer names.
composer_set = set()

# Iterate over each row and add composers to the set.
for composer_list in df_track_id_unormalized_composer["normalized_composer_list"]:
    if composer_list is None:
        continue
    for composer in composer_list:
        composer_set.add(composer)
        
# Convert to a sorted list.
unique_composers = sorted(composer_set)

In [35]:
# Create a dictionary that will hold the unique composer names and their composer IDs.
composer_dict = {name: idx for idx, name in enumerate(unique_composers, start=1)}

In [36]:
# Create the normalized "composer" DataFrame.
df_normalized_db_composer_table = pd.DataFrame(
    [(composer_id, composer_name) for composer_name, composer_id in composer_dict.items()],
    columns=["composer_id", "composer_name"]
).sort_values("composer_id")

# Define the column datatypes.
schema = {
    "composer_id": "Int64",
    "composer_name": "string"
}

# Apply the datatypes.
df_normalized_db_composer_table = df_normalized_db_composer_table.astype(schema)

In [37]:
df_normalized_db_composer_table.head()

,composer_id,composer_name
0,1,A. Bouchard
1,2,A. F. Iommi
2,3,A. Isbell
3,4,A. Jamal
4,5,A. Jones


In [38]:
# Build the "composer_track" junction DataFrame/table:
# - Iterate over "df_track_id_unormalized_composer" row by row
# - Extract the "track_id" for the current row
# - Loop through the "normalized_composer_list" for that track
# - For each composer name, retrieve its composer ID from the "composer_dict" dictionary
# - Add a new entry ("composer_id", "track_id") to the junction table
rows = []

for index, row in df_track_id_unormalized_composer.iterrows():
    track_id = row["track_id"]
    composer_list = row["normalized_composer_list"]

    if composer_list is None:
        continue

    for composer_name in composer_list:
        composer_id = composer_dict.get(composer_name)

        # If the composer exists in the dictionary append the values to the rows list.
        if composer_id is not None:
            rows.append((composer_id, track_id))

# Create the "composer_track" junction DataFrame.
df_normalized_db_composer_track_table = pd.DataFrame(rows, columns=["composer_id", "track_id"])

In [39]:
# Insert a surrogate primary key column to the DataFrame.
df_normalized_db_composer_track_table.insert(
    0,
    "composer_track_id",
    range(1, len(df_normalized_db_composer_track_table) + 1)
)

In [40]:
# Edit the datatypes of the DataFrame's columns.

# Define the datatypes.
schema = {
    "composer_track_id": "Int64",
    "composer_id": "Int64",
    "track_id": "Int64",
}

# Apply the datatypes.
df_normalized_db_composer_track_table = df_normalized_db_composer_track_table.astype(schema)

In [41]:
df_normalized_db_composer_track_table.head()

,composer_track_id,composer_id,track_id
0,1,41,1
1,2,598,1
2,3,123,1
3,4,293,3
4,5,820,3


**Move on to normalize the "album" and "artist" DataFrames/tables and create the "artist_album" junction table.**

In [42]:
# Store the "album" entity into a new DataFrame.
df_original_db_album_table = pd.read_sql("SELECT * FROM album", engine_original_db)

df_original_db_album_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   album_id   347 non-null    str  
 1   title      347 non-null    str  
 2   artist_id  347 non-null    str  
dtypes: str(3)
memory usage: 17.7 KB


In [43]:
# Create the normalized "df_normalized_db_album_table" Dataframe, using the required data from the original DataFrame. 
df_normalized_db_album_table = pd.DataFrame().assign(album_id=df_original_db_album_table["album_id"], album_title=df_original_db_album_table["title"])

# Define the column datatypes.
schema = {
    "album_id": "Int64",
    "album_title": "string"
}

# Apply the datatypes.
df_normalized_db_album_table = df_normalized_db_album_table.astype(schema)
df_normalized_db_album_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   album_id     347 non-null    Int64 
 1   album_title  347 non-null    string
dtypes: Int64(1), string(1)
memory usage: 13.6 KB


In [44]:
# Store the "artist" entity into a new DataFrame.
df_original_db_artist_table = pd.read_sql("SELECT * FROM artist", engine_original_db)

df_original_db_artist_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   artist_id  275 non-null    str  
 1   name       275 non-null    str  
dtypes: str(2)
memory usage: 10.7 KB


In [45]:
# Create a new mapping dictionary that will handle all splits for irregular raw artist strings.
artist_manual_split_dict = {
    "AC/DC": ["AC/DC"],
    "Chico Science & Nação Zumbi": ["Chico Science", "Nação Zumbi"],
    "Frank Zappa & Captain Beefheart": ["Frank Zappa", "Captain Beefheart"],
    "Milton Nascimento & Bebeto": ["Milton Nascimento", "Paula E Bebeto"],
    "Pedro Luís & A Parede": ["Pedro Luís e a Parede"],
    "Edson, DJ Marky & DJ Patife Featuring Fernanda Porto": ["DJ Marky Edson", "DJ Patife", "Fernanda Porto"],
    "Santana Feat. Dave Matthews": ["Carlos Santana", "Dave Matthews"],
    "Santana Feat. Everlast": ["Carlos Santana", "Everlast"],
    "Santana Feat. Rob Thomas": ["Carlos Santana", "Rob Thomas"],
    "Santana Feat. Lauryn Hill & Cee-Lo": ["Carlos Santana", "Lauryn Hill", "Cee-Lo"],
    "Santana Feat. The Project G&B": ["Carlos Santana", "The Project G&B"],
    "Santana Feat. Maná": ["Carlos Santana", "Maná"],
    "Santana Feat. Eagle-Eye Cherry": ["Carlos Santana", "Eagle-Eye Cherry"],
    "Santana Feat. Eric Clapton": ["Carlos Santana", "Eric Clapton"],
    "Toquinho & Vinícius": ["Toquinho", "Vinícius de Moraes"],
    "Vinícius De Moraes & Baden Powell": ["Vinícius de Moraes", "Baden Powell"],
    "Vinícius E Qurteto Em Cy": ["Vinícius de Moraes", "Qurteto em Cy"],
    "Vinícius E Odette Lara": ["Vinícius de Moraes", "Odette Lara"],
    "Vinicius, Toquinho & Quarteto Em Cy": ["Vinícius de Moraes", "Toquinho", "Qurteto em Cy"],
    "Motörhead & Girlschool": ["Motörhead", "Girlschool"],
    "Page & Plant": ["Page & Plant"],
    "R.E.M. Feat. Kate Pearson": ["R.E.M.", "Kate Pearson"],
    "R.E.M. Feat. KRS-One": ["R.E.M.", "KRS-One"],
    "R.E.M.": ["R.E.M."],
    "Stevie Ray Vaughan & Double Trouble": ["Stevie Ray Vaughan", "Double Trouble"],
    "Terry Bozzio, Tony Levin & Steve Stevens": ["Terry Bozzio", "Tony Levin", "Steve Stevens"],
    "Christina Aguilera featuring BigElf": ["Christina Aguilera", "BigElf"],
    "Aerosmith & Sierra Leone's Refugee Allstars": ["Aerosmith", "Sierra Leone's Refugee Allstars"],
    "Dhani Harrison & Jakob Dylan": ["Dhani Harrison", "Jakob Dylan"],
    "Big & Rich": ["Big & Rich"],
    "Jack's Mannequin & Mick Fleetwood": ["Jack's Mannequin", "Mick Fleetwood"],
    "Gustavo & Andres Veiga & Salazar": ["Gustavo Veiga", "Andrés Salazar"],
    "Mundo Livre S/A": ["Mundo Livre S/A"],
    "DJ Dolores & Orchestra Santa Massa": ["DJ Dolores", "Orchestra Santa Massa"],
    "Sabotage E Instituto": ["Sabotage e Instituto"],
    "Habib Koité and Bamada": ["Habib Koité and Bamada"],
    "Luciana Souza/Romero Lubambo": ["Luciana Souza", "Romero Lubambo"],
    "Alberto Turco & Nova Schola Gregoriana": ["Alberto Turco", "Nova Schola Gregoriana"],
    "Richard Marlow & The Choir of Trinity College, Cambridge": ["Richard Marlow", "The Choir of Trinity College Cambridge"],
    "English Concert & Trevor Pinnock": ["English Concert & Trevor Pinnock"],
    "Anne-Sophie Mutter, Herbert Von Karajan & Wiener Philharmoniker": ["Anne-Sophie Mutter", "Herbert Von Karajan", "Wiener Philharmoniker"],
    "Hilary Hahn, Jeffrey Kahane, Los Angeles Chamber Orchestra & Margaret Batjer": ["Hilary Hahn", "Jeffrey Kahane", "Los Angeles Chamber Orchestra", "Margaret Batjer"],
    "Academy of St. Martin in the Fields & Sir Neville Marriner": ["Academy of St. Martin in the Fields", "Sir Neville Marriner"],
    "Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner": ["Academy of St. Martin in the Fields Chamber Ensemble", "Sir Neville Marriner"],
    "Berliner Philharmoniker, Claudio Abbado & Sabine Meyer": ["Berliner Philharmoniker", "Claudio Abbado", "Sabine Meyer"],
    "Royal Philharmonic Orchestra & Sir Thomas Beecham": ["Royal Philharmonic Orchestra", "Sir Thomas Beecham"],
    "Orchestre Révolutionnaire et Romantique & John Eliot Gardiner": ["Orchestre Révolutionnaire et Romantique", "John Eliot Gardiner"],
    "Britten Sinfonia, Ivor Bolton & Lesley Garrett": ["Britten Sinfonia", "Ivor Bolton", "Lesley Garrett"],
    "Chicago Symphony Chorus, Chicago Symphony Orchestra & Sir Georg Solti": ["Chicago Symphony Chorus", "Chicago Symphony Orchestra", "Sir Georg Solti"],
    "Sir Georg Solti & Wiener Philharmoniker": ["Sir Georg Solti", "Wiener Philharmoniker"],
    "Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair": ["Academy of St. Martin in the Fields", "John Birch", "Sir Neville Marriner", "Sylvia McNair"],
    "London Symphony Orchestra & Sir Charles Mackerras": ["London Symphony Orchestra", "Sir Charles Mackerras"],
    "Barry Wordsworth & BBC Concert Orchestra": ["Barry Wordsworth", "BBC Concert Orchestra"],
    "Herbert Von Karajan, Mirella Freni & Wiener Philharmoniker": ["Herbert Von Karajan", "Mirella Freni", "Wiener Philharmoniker"],
    "Leonard Bernstein & New York Philharmonic": ["Leonard Bernstein", "New York Philharmonic"],
    "Boston Symphony Orchestra & Seiji Ozawa": ["Boston Symphony Orchestra", "Seiji Ozawa"],
    "Aaron Copland & London Symphony Orchestra": ["Aaron Copland", "London Symphony Orchestra"],
    "Sergei Prokofiev & Yuri Temirkanov": ["Sergei Prokofiev", "Yuri Temirkanov"],
    "Chicago Symphony Orchestra & Fritz Reiner": ["Chicago Symphony Orchestra", "Fritz Reiner"],
    "Emanuel Ax, Eugene Ormandy & Philadelphia Orchestra": ["Emanuel Ax", "Eugene Ormandy", "Philadelphia Orchestra"],
    "Berliner Philharmoniker & Hans Rosbaud": ["Berliner Philharmoniker", "Hans Rosbaud"],
    "Academy of St. Martin in the Fields, Sir Neville Marriner & William Bennett": ["Academy of St. Martin in the Fields", "Sir Neville Marriner", "William Bennett"],
    "Felix Schmidt, London Symphony Orchestra & Rafael Frühbeck de Burgos": ["Felix Schmidt", "London Symphony Orchestra", "Rafael Frühbeck de Burgos"],
    "Edo de Waart & San Francisco Symphony": ["Edo de Waart", "San Francisco Symphony"],
    "Antal Doráti & London Symphony Orchestra": ["Antal Doráti", "London Symphony Orchestra"],
    "Choir Of Westminster Abbey & Simon Preston": ["Choir Of Westminster Abbey", "Simon Preston"],
    "Michael Tilson Thomas & San Francisco Symphony": ["Michael Tilson Thomas", "San Francisco Symphony"],
    "Chor der Wiener Staatsoper, Herbert Von Karajan & Wiener Philharmoniker": ["Chor der Wiener Staatsoper", "Herbert Von Karajan", "Wiener Philharmoniker"],
    "Berliner Philharmoniker & Herbert Von Karajan": ["Berliner Philharmoniker", "Herbert Von Karajan"],
    "Sir Georg Solti, Sumi Jo & Wiener Philharmoniker": ["Sir Georg Solti", "Sumi Jo", "Wiener Philharmoniker"],
    "Otto Klemperer & Philharmonia Orchestra": ["Otto Klemperer", "Philharmonia Orchestra"],
    "Philharmonia Orchestra & Sir Neville Marriner": ["Philharmonia Orchestra", "Sir Neville Marriner"],
    "Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart": ["Academy of St. Martin in the Fields", "Sir Neville Marriner", "Thurston Dart"],
    "Les Arts Florissants & William Christie": ["Les Arts Florissants", "William Christie"],
    "Adrian Leaper & Doreen de Feis": ["Adrian Leaper", "Doreen de Feis"],
    "Charles Dutoit & L'Orchestre Symphonique de Montréal": ["Charles Dutoit", "L'Orchestre Symphonique de Montréal"],
    "Equale Brass Ensemble, John Eliot Gardiner & Munich Monteverdi Orchestra and Choir": ["Equale Brass Ensemble", "John Eliot Gardiner", "Munich Monteverdi Orchestra and Choir"],
    "Kent Nagano and Orchestre de l'Opéra de Lyon": ["Kent Nagano", "Orchestre de l'Opéra de Lyon"],
    "Göteborgs Symfoniker & Neeme Järvi": ["Göteborgs Symfoniker", "Neeme Järvi"],
    "Mela Tenenbaum, Pro Musica Prague & Richard Kapp": ["Mela Tenenbaum", "Pro Musica Prague", "Richard Kapp"],
    "C. Monteverdi, Nigel Rogers - Chiaroscuro; London Baroque; London Cornett & Sackbu": ["Claudio Monteverdi", "Nigel Rogers", "Chiaroscuro", "London Baroque", "London Cornett", "Sackbut Ensemble"],
    "Various Artists": [],
}

In [46]:
# Create a canonicalized dictionary for the artist names.
canonicalized_artist_manual_split_dict = {canonicalize_manual_string(raw_artist_string): artist_manual_split_dict[raw_artist_string] for raw_artist_string in artist_manual_split_dict}

In [47]:
# Merge the original db tables "artist" and "album", in order to split the non-atomic
# artist strings while keeping track of the relation between each album and its artist/s.
df_album_unormalized_artist = df_original_db_album_table.merge(
    df_original_db_artist_table,
    on="artist_id",
    how="left"
)

In [48]:
# Rename the column names of the above DataFrame.
df_album_unormalized_artist.columns = ["album_id", "album_title", "artist_id", "unormalized_artist"]

df_album_unormalized_artist.info()

<class 'pandas.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   album_id            347 non-null    str  
 1   album_title         347 non-null    str  
 2   artist_id           347 non-null    str  
 3   unormalized_artist  347 non-null    str  
dtypes: str(4)
memory usage: 26.4 KB


In [49]:
# Create a function that will handle the raw artist strings whether they appear on the mapping dictionary or not.
def process_artist_string(s):
    # Handle missing values
    if s is None or pd.isna(s):
        return None

    canonical = canonicalize_manual_string(s)
    
    # If the string is in the manual mapping dictionary, return a message.
    if canonical in canonicalized_artist_manual_split_dict:
        return "Artist string handled by the mapping dictionary."
    # Otherwise, normalize the separators and return the normalized string.
    return normalize_separators(s)
    
# Apply the above function and create a new column in the DataFrame.
df_album_unormalized_artist["normalized_separators"] = (
    df_album_unormalized_artist["unormalized_artist"]
    .apply(process_artist_string)
)

In [50]:
df_album_unormalized_artist.head()

,album_id,album_title,artist_id,unormalized_artist,normalized_separators
0,1,For Those About To Rock We Salute You,1,AC/DC,Artist string handled by the mapping dictionary.
1,2,Balls to the Wall,2,Accept,Accept
2,3,Restless and Wild,2,Accept,Accept
3,4,Let There Be Rock,1,AC/DC,Artist string handled by the mapping dictionary.
4,5,Big Ones,3,Aerosmith,Aerosmith


In [51]:
# Add a new column to the DataFrame, that contains a list with the artists of each album per row.
def album_artist_list(s):
    if s is None or pd.isna(s):
        return None
        
    # Canonicalize the input.
    canonical = canonicalize_manual_string(s)

    if canonical in canonicalized_artist_manual_split_dict:
        result = canonicalized_artist_manual_split_dict[canonical]
    else:
        normalized = normalize_separators(s)
        result = normalized.split("||")

    # Return "None" when we meet an empty list.
    return result if result else None


df_album_unormalized_artist["album_artist_list"] = (
    df_album_unormalized_artist["unormalized_artist"]
    .apply(album_artist_list)
)

In [52]:
df_album_unormalized_artist.head()

,album_id,album_title,artist_id,unormalized_artist,normalized_separators,album_artist_list
0,1,For Those About To Rock We Salute You,1,AC/DC,Artist string handled by the mapping dictionary.,[AC/DC]
1,2,Balls to the Wall,2,Accept,Accept,[Accept]
2,3,Restless and Wild,2,Accept,Accept,[Accept]
3,4,Let There Be Rock,1,AC/DC,Artist string handled by the mapping dictionary.,[AC/DC]
4,5,Big Ones,3,Aerosmith,Aerosmith,[Aerosmith]


In [53]:
# Create a function that will normalize each atomic atomic name using the previously created "normalize_atomic_name" function.
def normalize_artist_list(lst):
    # Handle missing values.
    if lst is None or lst is pd.NA:
        return None
    
    cleaned = []
    
    for name in lst:
        atomic = normalize_atomic_name(name)
        
        # Skip unwanted role words
        if atomic.lower() in excluded_words:
            continue
        
        cleaned.append(atomic)
    
    return cleaned

df_album_unormalized_artist["normalized_artist_list"] = (
    df_album_unormalized_artist["album_artist_list"]
    .apply(normalize_artist_list)
)

In [54]:
df_album_unormalized_artist.head()

,album_id,album_title,artist_id,unormalized_artist,normalized_separators,album_artist_list,normalized_artist_list
0,1,For Those About To Rock We Salute You,1,AC/DC,Artist string handled by the mapping dictionary.,[AC/DC],[AC/DC]
1,2,Balls to the Wall,2,Accept,Accept,[Accept],[Accept]
2,3,Restless and Wild,2,Accept,Accept,[Accept],[Accept]
3,4,Let There Be Rock,1,AC/DC,Artist string handled by the mapping dictionary.,[AC/DC],[AC/DC]
4,5,Big Ones,3,Aerosmith,Aerosmith,[Aerosmith],[Aerosmith]


In [55]:
# Create a set to collect unique artist names.
artist_set = set()

# Iterate over each row and add artists to the set.
for artist_list in df_album_unormalized_artist["normalized_artist_list"]:
    if artist_list is None:
        continue
    for artist in artist_list:
        artist_set.add(artist)
        
# Convert to a sorted list.
unique_artists = sorted(artist_set)

In [56]:
# Create a dictionary that will hold the unique artist names and their corresponding artist IDs.
artist_dict = {name: idx for idx, name in enumerate(unique_artists, start=1)}

In [57]:
# Create the normalized "artist" DataFrame.
df_normalized_db_artist_table = pd.DataFrame(
    [(artist_id, artist_name) for artist_name, artist_id in artist_dict.items()],
    columns=["artist_id", "artist_name"]
).sort_values("artist_id")

# Define the column datatypes.
schema = {
    "artist_id": "Int64",
    "artist_name": "string"
}

# Apply the datatypes.
df_normalized_db_artist_table = df_normalized_db_artist_table.astype(schema)

In [58]:
df_normalized_db_artist_table.head()

,artist_id,artist_name
0,1,AC/DC
1,2,Aaron Copland
2,3,Aaron Goldberg
3,4,Academy Of St. Martin In The Fields
4,5,Academy Of St. Martin In The Fields Chamber En...


In [59]:
# Build the artist_album junction DataFrame/table:
# - Iterate over "df_album_unormalized_artist" row by row
# - Extract the "album_id" for the current row
# - Loop through the "normalized_artist_list" for that album
# - For each artist name, retrieve its artist ID from the "artist_dict' dictionary
# - Add a new entry ("artist_id", "album_id") to the junction table
rows = []

for index, row in df_album_unormalized_artist.iterrows():
    album_id = row["album_id"]
    artist_list = row["normalized_artist_list"]

    # Skip missing or invalid values.
    if not isinstance(artist_list, list):
        continue

    for artist_name in artist_list:
        artist_id = artist_dict.get(artist_name)
        # If the artist exists in the dictionary append the values to the rows list.
        if artist_id is not None:
            rows.append((artist_id, album_id))


# Create the "artist_album" junction DataFrame.
df_normalized_db_artist_album_table = pd.DataFrame(rows, columns=["artist_id", "album_id"])

In [60]:
# Insert a surrogate primary key column to the DataFrame.
df_normalized_db_artist_album_table.insert(
    0,
    "artist_album_id",
    range(1, len(df_normalized_db_artist_album_table) + 1)
)

In [61]:
# Edit the datatypes of the DataFrame's columns.

# Define the datatypes.
schema = {
    "artist_album_id": "Int64",
    "artist_id": "Int64",
    "album_id": "Int64",
}

# Apply the datatypes.
df_normalized_db_artist_album_table = df_normalized_db_artist_album_table.astype(schema)

In [62]:
df_normalized_db_artist_album_table.head()

,artist_album_id,artist_id,album_id
0,1,1,1
1,2,6,2
2,3,6,3
3,4,1,4
4,5,8,5


In [63]:
# Create the "album-track" junction DataFrame/table.
df_normalized_db_album_track_table = (
    df_original_db_track_table[["album_id", "track_id"]]
)

In [64]:
# Insert a surrogate primary key column to the DataFrame.
df_normalized_db_album_track_table.insert(
    0,
    "album_track_id",
    range(1, len(df_normalized_db_album_track_table) + 1)
)

In [65]:
# Edit the datatypes of the DataFrame's columns.

# Define the datatypes.
schema = {
    "album_track_id": "Int64",
    "album_id": "Int64",
    "track_id": "Int64",
}

# Apply the datatypes.
df_normalized_db_album_track_table = df_normalized_db_album_track_table.astype(schema)

In [66]:
df_normalized_db_album_track_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 3503 entries, 0 to 3502
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   album_track_id  3503 non-null   Int64
 1   album_id        3503 non-null   Int64
 2   track_id        3503 non-null   Int64
dtypes: Int64(3)
memory usage: 92.5 KB


In [67]:
df_normalized_db_album_track_table.head()

,album_track_id,album_id,track_id
0,1,1,1
1,2,2,2
2,3,3,3
3,4,3,4
4,5,3,5


In [68]:
# After finalizing the DataFrames that relate to the original unnormalized "track" table,
# we can now build the DataFrame that will hold the normalized values for the "track" table
# based on "df_normalized_db_track_table".
df_normalized_db_track_table = (
    df_original_db_track_table[["track_id", "media_type_id", "name", "milliseconds", "bytes", "unit_price"]]
    .rename(columns={
        "name": "track_name",
        "milliseconds": "duration_milliseconds",
        "bytes": "size_bytes",
        "unit_price": "unit_price_cents"
    })
    .assign(
        unit_price_cents=lambda x: (x["unit_price_cents"] * 100)
            .round()
            .astype("Int64")
    )
)

In [69]:
df_normalized_db_track_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 3503 entries, 0 to 3502
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   track_id               3503 non-null   Int64 
 1   media_type_id          3503 non-null   Int64 
 2   track_name             3503 non-null   string
 3   duration_milliseconds  3503 non-null   Int64 
 4   size_bytes             3503 non-null   Int64 
 5   unit_price_cents       3503 non-null   Int64 
dtypes: Int64(5), string(1)
memory usage: 236.1 KB


In [70]:
# Create the final "playlist" DataFrame.

# Import the "playlist" table.
df_original_db_playlist_table = pd.read_sql("SELECT * FROM playlist", engine_original_db)

# Create a new DataFrame that will contain the final version of the table, renaming its columns when necessary.
df_normalized_db_playlist_table = df_original_db_playlist_table.rename(
    columns={
        "name": "playlist_name"
    }
).copy()

# Define the datatypes.
schema = {
    "playlist_id": "Int64",
    "playlist_name": "string",
}

# Apply the datatypes.
df_normalized_db_playlist_table = df_normalized_db_playlist_table.astype(schema)

In [71]:
df_normalized_db_playlist_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   playlist_id    18 non-null     Int64 
 1   playlist_name  18 non-null     string
dtypes: Int64(1), string(1)
memory usage: 657.0 bytes


In [72]:
# Create the final "playlist_track" DataFrame.

# Import the "playlist_track" table.
df_original_db_playlist_track_table = pd.read_sql("SELECT * FROM playlist_track", engine_original_db)

In [73]:
df_original_db_playlist_track_table.head()

,playlist_id,track_id
0,1,3402
1,1,3389
2,1,3390
3,1,3391
4,1,3392


In [74]:
# Create a copy of the original DataFrame, then insert a surrogate primary key column to the DataFrame.
df_normalized_db_playlist_track_table = df_original_db_playlist_track_table.copy()

df_normalized_db_playlist_track_table.insert(
    0,
    "playlist_track_id",
    range(1, len(df_normalized_db_playlist_track_table) + 1)
)

# Define the datatypes.
schema = {
    "playlist_track_id": "Int64",
    "playlist_id": "Int64",
    "track_id": "Int64",
}

# Apply the datatypes.
df_normalized_db_playlist_track_table = df_normalized_db_playlist_track_table.astype(schema)

In [75]:
df_normalized_db_playlist_track_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 8715 entries, 0 to 8714
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   playlist_track_id  8715 non-null   Int64
 1   playlist_id        8715 non-null   Int64
 2   track_id           8715 non-null   Int64
dtypes: Int64(3)
memory usage: 229.9 KB


In [76]:
# Create the final "invoice" DataFrame.

# Import the original "invoice" table.
df_original_db_invoice_table = pd.read_sql("SELECT * FROM invoice", engine_original_db)

# Build the normalized "invoice" DataFrame.
df_normalized_db_invoice_table = (
    df_original_db_invoice_table[
        [
            "invoice_id",
            "customer_id",
            "invoice_date",
            "billing_address",
            "billing_city",
            "billing_state",
            "billing_country",
            "billing_postal",
            "total"
        ]
    ]
    .rename(columns={
        "invoice_date": "invoice_datetime",
        "billing_state": "billing_administrative_division",
        "total": "invoice_total_cents"
    })
    .assign(
        # Convert to UTC timestamp.
        invoice_datetime=lambda x: pd.to_datetime(x["invoice_datetime"]).dt.tz_localize("UTC"),

        # Convert DOUBLE PRECISION to integer cents.
        invoice_total_cents=lambda x: (x["invoice_total_cents"] * 100)
            .round()
            .astype("Int64")
    )
)

# Apply the final schema.
schema = {
    "invoice_id": "Int64",
    "customer_id": "Int64",
    "invoice_datetime": "datetime64[ns, UTC]",
    "billing_address": "string",
    "billing_city": "string",
    "billing_administrative_division": "string",
    "billing_country": "string",
    "billing_postal": "string",
    "invoice_total_cents": "Int64"
}

df_normalized_db_invoice_table = df_normalized_db_invoice_table.astype(schema)

In [77]:
df_normalized_db_invoice_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype              
---  ------                           --------------  -----              
 0   invoice_id                       614 non-null    Int64              
 1   customer_id                      614 non-null    Int64              
 2   invoice_datetime                 614 non-null    datetime64[ns, UTC]
 3   billing_address                  614 non-null    string             
 4   billing_city                     614 non-null    string             
 5   billing_administrative_division  614 non-null    string             
 6   billing_country                  614 non-null    string             
 7   billing_postal                   614 non-null    string             
 8   invoice_total_cents              614 non-null    Int64              
dtypes: Int64(3), datetime64[ns, UTC](1), string(5)
memory usage: 70.0 KB


In [78]:
df_normalized_db_invoice_table.head()

,invoice_id,customer_id,invoice_datetime,billing_address,billing_city,billing_administrative_division,billing_country,billing_postal,invoice_total_cents
0,1,18,2017-01-03 00:00:00+00:00,627 Broadway,New York,NY,USA,10012-2612,1584
1,2,30,2017-01-03 00:00:00+00:00,230 Elgin Street,Ottawa,ON,Canada,K2P 1L7,990
2,3,40,2017-01-05 00:00:00+00:00,"8, Rue Hanovre",Paris,None,France,75002,198
3,4,18,2017-01-06 00:00:00+00:00,627 Broadway,New York,NY,USA,10012-2612,792
4,5,27,2017-01-07 00:00:00+00:00,1033 N Park Ave,Tucson,AZ,USA,85719,1683


In [79]:
# Create the final "invoice_line" DataFrame.

# Import the original "invoice_line" table.
df_original_db_invoice_line_table = pd.read_sql("SELECT * FROM invoice_line", engine_original_db)

# Build the normalized "invoice_line" DataFrame.
df_normalized_db_invoice_line_table = (
    df_original_db_invoice_line_table[
        [
            "invoice_line_id",
            "invoice_id",
            "track_id",
            "unit_price",
            "quantity"
        ]
    ]
    .rename(columns={
        "unit_price": "unit_price_cents"
    })
    .assign(
        # Convert DOUBLE PRECISION to integer cents.
        unit_price_cents=lambda x: (x["unit_price_cents"] * 100)
            .round()
            .astype("Int64"),

        # Convert DOUBLE PRECISION quantity to integer.
        quantity=lambda x: x["quantity"]
            .round()
            .astype("Int64")
    )
)

# Apply the final schema.
schema = {
    "invoice_line_id": "Int64",
    "invoice_id": "Int64",
    "track_id": "Int64",
    "unit_price_cents": "Int64",
    "quantity": "Int64"
}

df_normalized_db_invoice_line_table = df_normalized_db_invoice_line_table.astype(schema)

In [80]:
df_normalized_db_invoice_line_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 4757 entries, 0 to 4756
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   invoice_line_id   4757 non-null   Int64
 1   invoice_id        4757 non-null   Int64
 2   track_id          4757 non-null   Int64
 3   unit_price_cents  4757 non-null   Int64
 4   quantity          4757 non-null   Int64
dtypes: Int64(5)
memory usage: 209.2 KB


In [81]:
df_normalized_db_invoice_line_table.head()

,invoice_line_id,invoice_id,track_id,unit_price_cents,quantity
0,1,1,1158,99,1
1,2,1,1159,99,1
2,3,1,1160,99,1
3,4,1,1161,99,1
4,5,1,1162,99,1


**We can now move on to the creation of the DataFrames that will handle addresses in the normalized database.**

In [82]:
# Create the "country" DataFrame.

# Since the original database includes only 24 countries, we handle manually the necessary info.
df_normalized_db_country_table = pd.DataFrame(
    [
        [1, "Argentina", "ARG"],
        [2, "Australia", "AUS"],
        [3, "Austria", "AUT"],
        [4, "Belgium", "BEL"],
        [5, "Brazil", "BRA"],
        [6, "Canada", "CAN"],
        [7, "Chile", "CHL"],
        [8, "Czechia", "CZE"],
        [9, "Denmark", "DNK"],
        [10, "Finland", "FIN"],
        [11, "France", "FRA"],
        [12, "Germany", "DEU"],
        [13, "Hungary", "HUN"],
        [14, "India", "IND"],
        [15, "Ireland", "IRL"],
        [16, "Italy", "ITA"],
        [17, "Netherlands", "NLD"],
        [18, "Norway", "NOR"],
        [19, "Poland", "POL"],
        [20, "Portugal", "PRT"],
        [21, "Spain", "ESP"],
        [22, "Sweden", "SWE"],
        [23, "United Kingdom", "GBR"],
        [24, "USA", "USA"],
        
    ],
    columns=["country_id", "country_name", "country_iso_code"]
)

df_normalized_db_country_table = df_normalized_db_country_table.astype({
    "country_id": "Int64",
    "country_name": "string",
    "country_iso_code": "string"
})

In [83]:
df_normalized_db_country_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   country_id        24 non-null     Int64 
 1   country_name      24 non-null     string
 2   country_iso_code  24 non-null     string
dtypes: Int64(1), string(2)
memory usage: 970.0 bytes


In [84]:
# Create the "postal_zip_code" DataFrame.

# Start by importing the postal/zip codes and corresponding country that relates to each employee and customer.

# Import the original "employee" table.
df_original_db_employee_table = pd.read_sql("SELECT * FROM employee", engine_original_db)

# Import the original "customer" table.
df_original_db_customer_table = pd.read_sql("SELECT * FROM customer", engine_original_db)

# Trim whitespaces from CHAR columns of the imported tables.
df_original_db_employee_table["first_name"] = df_original_db_employee_table["first_name"].str.strip()
df_original_db_employee_table["last_name"] = df_original_db_employee_table["last_name"].str.strip()
df_original_db_customer_table["first_name"] = df_original_db_customer_table["first_name"].str.strip()
df_original_db_customer_table["last_name"] = df_original_db_customer_table["last_name"].str.strip()

# Create two DataFrames that contain the needed information for the normalized "postal_zip_code" df.
# Rename the column containing the countries, so that we simplify the merging of the DataFrames.
employee_postal_df = df_original_db_employee_table[["country", "postal_code"]].rename(
    columns={
        "country": "country_name",
        "postal_code": "code"
    }
)

customer_postal_df = df_original_db_customer_table[["country", "postal_code"]].rename(
    columns={
        "country": "country_name",
        "postal_code": "code"
    }
)

In [85]:
# Join the two DataFrames using ".concat()", taking into account that we need to replace "Czech Republic" from the original database, with "Czechia".
country_name_fix = {
    "Czech Republic": "Czechia"
}

postal_zip_code_df = (
    pd.concat([employee_postal_df, customer_postal_df], ignore_index=True)
      .assign(country_name=lambda df: df["country_name"].replace(country_name_fix))
      .drop_duplicates()
)

In [86]:
# Use ".merge()" to add the country IDs. 
postal_zip_code_df = postal_zip_code_df.merge(
    df_normalized_db_country_table[["country_name", "country_id"]],
    on="country_name",
    how="left"
)

In [87]:
# Add a surrogate PK column to the DataFrame.
postal_zip_code_df["postal_zip_code_id"] = range(1, len(postal_zip_code_df) + 1)

In [88]:
postal_zip_code_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   country_name        66 non-null     object
 1   code                63 non-null     str   
 2   country_id          66 non-null     Int64 
 3   postal_zip_code_id  66 non-null     int64 
dtypes: Int64(1), int64(1), object(1), str(1)
memory usage: 2.6+ KB


In [89]:
# Create the normalized "postal_zip_code" DataFrame.
df_normalized_db_postal_zip_code_table = postal_zip_code_df[["postal_zip_code_id", "country_id", "code"]]

# Change the DataFrame's datatypes.
df_normalized_db_postal_zip_code_table = df_normalized_db_postal_zip_code_table.astype({
    "postal_zip_code_id": 'Int64',
    "country_id": 'Int64',
    "code": 'string'
})

In [90]:
df_normalized_db_postal_zip_code_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   postal_zip_code_id  66 non-null     Int64 
 1   country_id          66 non-null     Int64 
 2   code                63 non-null     string
dtypes: Int64(2), string(1)
memory usage: 2.2 KB


In [91]:
df_normalized_db_postal_zip_code_table.head()

,postal_zip_code_id,country_id,code
0,1,6,T5K 2N1
1,2,6,T2P 2T3
2,3,6,T2P 5M5
3,4,6,T2P 5G3
4,5,6,T3B 1Y7


In [92]:
# Create the "administrative_division" DataFrame.

# Since the original database includes only 50 administrative divisions, we handle manually the necessary info.
df_normalized_db_administrative_division_table = pd.DataFrame(
    [
        [1, 1, "Ciudad Autónoma de Buenos Aires", "Autonomous City"],
        [2, 2, "New South Wales", "State"],
        [3, 3, "Wien", "State"],
        [4, 4, "Brussels Hoofdstedelijk Gewest", "Region"],
        [5, 5, "São Paulo", "State"],
        [6, 5, "Rio de Janeiro", "State"],
        [7, 5, "Distrito Federal", "Federal District"],
        [8, 6, "Nova Scotia", "Province"],
        [9, 6, "Ontario", "Province"],
        [10, 6, "Quebec", "Province"],
        [11, 6, "Alberta", "Province"],
        [12, 6, "British Columbia", "Province"],
        [13, 6, "Northwest Territories", "Territory"],
        [14, 6, "Manitoba", "Province"],
        [15, 7, "Metropolitana", "Region"],
        [16, 8, "Praha", "Region"],
        [17, 9, "Hovedstaden", "Region"],
        [18, 10, "Uusimaa", "Region"],
        [19, 11, "Île-de-France", "Region"],
        [20, 11, "Rhône-Alpes", "Region"],
        [21, 11, "Aquitaine", "Region"],
        [22, 11, "Bourgogne", "Region"],
        [23, 12, "Berlin", "State"],
        [24, 12, "Baden-Württemberg", "State"],
        [25, 12, "Hessen", "State"],
        [26, 13, "Budapest", "County"],
        [27, 14, "Delhi", "Union Territory"],
        [28, 14, "Karnataka", "State"],
        [29, 15, "Dublin", "County"],
        [30, 16, "Lazio", "Region"],
        [31, 17, "Noord‑Holland", "Province"],
        [32, 18, "Oslo", "County"],
        [33, 19, "Masovian", "Province"],
        [34, 20, "Porto", "District"],
        [35, 20, "Lisboa", "District"],
        [36, 21, "Comunidad de Madrid", "Autonomous Community"],
        [37, 22, "Stockholm", "County"],
        [38, 23, "Greater London", "Ceremonial County"],
        [39, 23, "City of Edinburgh", "Council Area"],
        [40, 24, "Arizona", "State"],
        [41, 24, "California", "State"],
        [42, 24, "Florida", "State"],
        [43, 24, "Illinois", "State"],
        [44, 24, "Massachusetts", "State"],
        [45, 24, "Nevada", "State"],
        [46, 24, "New York", "State"],
        [47, 24, "Texas", "State"],
        [48, 24, "Utah", "State"],
        [49, 24, "Washington", "State"],
        [50, 24, "Wisconsink", "State"],
    ],
    columns=["administrative_division_id", "country_id", "division_name", "division_type"]
)

df_normalized_db_administrative_division_table = df_normalized_db_administrative_division_table.astype({
    "administrative_division_id": "Int64",
    "country_id": "Int64",
    "division_name": "string",
    "division_type": "string"
})

In [93]:
df_normalized_db_administrative_division_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   administrative_division_id  50 non-null     Int64 
 1   country_id                  50 non-null     Int64 
 2   division_name               50 non-null     string
 3   division_type               50 non-null     string
dtypes: Int64(2), string(2)
memory usage: 2.7 KB


In [94]:
# Create the "city" DataFrame.

# Since the original database includes only 55 cities, we handle manually the necessary info.
df_normalized_db_city_table = pd.DataFrame(
    [
        [1, 31, "Amsterdam"],
        [2, 28, "Bangalore"],
        [3, 23, "Berlin"],
        [4, 21, "Bordeaux"],
        [5, 44, "Boston"],
        [6, 7, "Brasília"],
        [7, 4, "Brussels"],
        [8, 26, "Budapest"],
        [9, 1, "Buenos Aires"],
        [10, 11, "Calgary"],
        [11, 43, "Chicago"],
        [12, 17, "Copenhagen"],
        [13, 41, "Cupertino"],
        [14, 27, "Delhi"],
        [15, 22, "Dijon"],
        [16, 29, "Dublin"],
        [17, 39, "Edinburgh"],
        [18, 11, "Edmonton"],
        [19, 47, "Fort Worth"],
        [20, 25, "Frankfurt"],
        [21, 8, "Halifax"],
        [22, 18, "Helsinki"],
        [23, 11, "Lethbridge"],
        [24, 35, "Lisbon"],
        [25, 38, "London"],
        [26, 20, "Lyon"],
        [27, 50, "Madison"],
        [28, 36, "Madrid"],
        [29, 10, "Montréal"],
        [30, 41, "Mountain View"],
        [31, 46, "New York"],
        [32, 42, "Orlando"],
        [33, 32, "Oslo"],
        [34, 9, "Ottawa"],
        [35, 19, "Paris"],
        [36, 34, "Porto"],
        [37, 16, "Prague"],
        [38, 49, "Redmond"],
        [39, 45, "Reno"],
        [40, 6, "Rio de Janeiro"],
        [41, 30, "Rome"],
        [42, 48, "Salt Lake City"],
        [43, 15, "Santiago"],
        [44, 5, "São José dos Campos"],
        [45, 5, "São Paulo"],
        [46, 2, "Sidney"],
        [47, 37, "Stockholm"],
        [48, 24, "Stuttgart"],
        [49, 9, "Toronto"],
        [50, 40, "Tucson"],
        [51, 12, "Vancouver"],
        [52, 3, "Vienne"],
        [53, 33, "Warsaw"],
        [54, 14, "Winnipeg"],
        [55, 13, "Yellowknife"],
    ],
    columns=["city_id", "administrative_division_id",  "city_name"]
)

df_normalized_db_city_table = df_normalized_db_city_table.astype({
    "city_id": "Int64",
    "administrative_division_id": "Int64",
    "city_name": "string",
})

In [95]:
df_normalized_db_city_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   city_id                     55 non-null     Int64 
 1   administrative_division_id  55 non-null     Int64 
 2   city_name                   55 non-null     string
dtypes: Int64(2), string(1)
memory usage: 2.0 KB


In [96]:
# Create the "address" DataFrame.

# We previously imported the "employee" and "customer" entities
# from the original database and stored them in the following two DataFrames:
# "df_original_db_employee_table" & "df_original_db_customer_table".

# Create two new Dataframes (one for the addresses of customers and one for the addresses of the employees),
# that will be later used to create the final "address" Dataframe.
employee_address_df = df_original_db_employee_table[["city", "postal_code", "address"]].rename(
    columns={
        "postal_code": "postal_zip_code",
        "address": "address_line_1"
    }
)

customer_address_df = df_original_db_customer_table[["city", "postal_code", "address"]].rename(
    columns={
        "postal_code": "postal_zip_code",
        "address": "address_line_1"
    }
)

# Combine the two DataFrames.
combined_address_df = pd.concat([employee_address_df, customer_address_df], ignore_index=True)

# Create the "address_line_2" column in the new DataFrame.
combined_address_df["address_line_2"] = None

In [97]:
combined_address_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   city             68 non-null     str   
 1   postal_zip_code  64 non-null     str   
 2   address_line_1   68 non-null     str   
 3   address_line_2   0 non-null      object
dtypes: object(1), str(3)
memory usage: 4.4+ KB


In [98]:
# Use ".strip() in order to fix a trailing error on the city "Edinburgh ".
combined_address_df["city"] = combined_address_df["city"].str.strip()

# Use ".merge()" to extract the city and postal_zip_code IDs from the "df_normalized_db_city_table" and
# "df_normalized_db_postal_zip_code_table" DataFrames and store them as new columns in "combined_address_df".
combined_address_df = combined_address_df.merge(
    df_normalized_db_city_table[["city_name", "city_id"]],
    left_on="city",
    right_on="city_name",
    how="left"
).drop(columns=["city_name"])

combined_address_df = combined_address_df.merge(
    df_normalized_db_postal_zip_code_table[["code", "postal_zip_code_id"]],
    left_on="postal_zip_code",
    right_on="code",
    how="left"
).drop(columns=["code"])

In [99]:
combined_address_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   city                76 non-null     object
 1   postal_zip_code     64 non-null     object
 2   address_line_1      76 non-null     str   
 3   address_line_2      0 non-null      object
 4   city_id             76 non-null     Int64 
 5   postal_zip_code_id  76 non-null     Int64 
dtypes: Int64(2), object(3), str(1)
memory usage: 5.2+ KB


In [100]:
# Replace all missling values with None.
combined_address_df = combined_address_df.where(combined_address_df.notna(), None)

In [101]:
combined_address_df.head()

,city,postal_zip_code,address_line_1,address_line_2,city_id,postal_zip_code_id
0,Edmonton,T5K 2N1,11120 Jasper Ave NW,None,18,1
1,Calgary,T2P 2T3,825 8 Ave SW,None,10,2
2,Calgary,T2P 5M5,1111 6 Ave SW,None,10,3
3,Calgary,T2P 5G3,683 10 Street SW,None,10,4
4,Calgary,T3B 1Y7,7727B 41 Ave,None,10,5


In [102]:
# Drop the no longer required columns.
combined_address_df = combined_address_df.drop(columns=["city", "postal_zip_code"])

# Check the DataFrame.
combined_address_df.head()

,address_line_1,address_line_2,city_id,postal_zip_code_id
0,11120 Jasper Ave NW,None,18,1
1,825 8 Ave SW,None,10,2
2,1111 6 Ave SW,None,10,3
3,683 10 Street SW,None,10,4
4,7727B 41 Ave,None,10,5


In [103]:
# Rearrange the column order.
combined_address_df = combined_address_df[
    ["city_id", "postal_zip_code_id", "address_line_1", "address_line_2"]
]

# Create the final "address" DataFrame.
df_normalized_db_address_table = combined_address_df.copy()

df_normalized_db_address_table.insert(
    0,                      # position (first column)
    "address_id",           # new column name
    range(1, len(df_normalized_db_address_table) + 1)
)

In [104]:
# Change the datatypes.
df_normalized_db_address_table = df_normalized_db_address_table.astype({
    "address_id": "Int64",
    "address_line_1": "string",
    "address_line_2": "string"
})

In [105]:
df_normalized_db_address_table.head()

,address_id,city_id,postal_zip_code_id,address_line_1,address_line_2
0,1,18,1,11120 Jasper Ave NW,<NA>
1,2,10,2,825 8 Ave SW,<NA>
2,3,10,3,1111 6 Ave SW,<NA>
3,4,10,4,683 10 Street SW,<NA>
4,5,10,5,7727B 41 Ave,<NA>


In [106]:
# Create the "employee_title" and "employee_level" DataFrames.

# Since the original database includes only 6 employee titles and levels, we handle manually the necessary info.
df_normalized_db_employee_title_table = pd.DataFrame(
    [
        [1, "General Manager"],
        [2, "IT Manager"],
        [3, "IT Staff"],
        [4, "Sales Manager"],
        [5, "Sales Support Agent"],
        [6, "Senior General Manager"],
    ],
    columns=["employee_title_id", "employee_title_name"]
)

df_normalized_db_employee_title_table = df_normalized_db_employee_title_table.astype({
    "employee_title_id": "Int64",
    "employee_title_name": "string",
})


df_normalized_db_employee_level_table = pd.DataFrame(
    [
        [1, "L1"],
        [2, "L2"],
        [3, "L3"],
        [4, "L4"],
        [5, "L5"],
        [6, "L6"],
    ],
    columns=["employee_level_id", "employee_level_code"]
)

df_normalized_db_employee_level_table = df_normalized_db_employee_level_table.astype({
    "employee_level_id": "Int64",
    "employee_level_code": "string",
})

In [107]:
df_normalized_db_employee_title_table.info()
df_normalized_db_employee_level_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   employee_title_id    6 non-null      Int64 
 1   employee_title_name  6 non-null      string
dtypes: Int64(1), string(1)
memory usage: 321.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   employee_level_id    6 non-null      Int64 
 1   employee_level_code  6 non-null      string
dtypes: Int64(1), string(1)
memory usage: 246.0 bytes


In [108]:
# Create the normalized "employee" DataFrame.

# Use the previously imported "employee" entity from the original database,
# to copy some of the needed columns in the normalized "employee" DataFrame.
df_normalized_db_employee_table = df_original_db_employee_table[["employee_id", "reports_to", "last_name", "first_name", "birthdate", "hire_date", "phone", "fax", "email"]].rename(
    columns={
        "reports_to": "manager_id",
        "last_name": "employee_last_name",
        "first_name": "employee_first_name",
        "birthdate": "employee_birthdate",
        "hire_date": "employee_hiredate",
        "phone": "employee_phone",
        "fax": "employee_fax",
        "email": "employee_email",
    }
)

In [109]:
df_normalized_db_employee_table.head(9)

,employee_id,manager_id,employee_last_name,employee_first_name,employee_birthdate,employee_hiredate,employee_phone,employee_fax,employee_email
0,1,9,Adams,Andrew,1962-02-18,2016-08-14,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,1,Edwards,Nancy,1958-12-08,2016-05-01,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,2,Peacock,Jane,1973-08-29,2017-04-01,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,2,Park,Margaret,1947-09-19,2017-05-03,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,2,Johnson,Steve,1965-03-03,2017-10-17,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com
5,6,1,Mitchell,Michael,1973-07-01,2016-10-17,+1 (403) 246-9887,+1 (403) 246-9899,michael@chinookcorp.com
6,7,6,King,Robert,1970-05-29,2017-01-02,+1 (403) 456-9986,+1 (403) 456-8485,robert@chinookcorp.com
7,8,6,Callahan,Laura,1968-01-09,2017-03-04,+1 (403) 467-3351,+1 (403) 467-8772,laura@chinookcorp.com
8,9,NaN,Madan,Mohan,1961-01-26,2016-01-14,+1 (780) 428-9482,+1 (780) 428-3457,madan.mohan@chinookcorp.com


In [110]:
# Manually add the rest of the needed columns in the normalized "employee" DataFrame.
df_normalized_db_employee_table["employee_level_id"] = [5, 4, 1, 1, 1, 3, 2, 2, 6]
df_normalized_db_employee_table["employee_title_id"] = [1, 4, 5, 5, 5, 2, 3, 3, 6]
df_normalized_db_employee_table["address_id"] = [1, 2, 3, 4, 5, 6, 7, 8, 9]

In [111]:
# Rearrange the order of the columns.
df_normalized_db_employee_table = df_normalized_db_employee_table[
    [
        "employee_id",
        "employee_level_id",
        "employee_title_id",
        "address_id",
        "manager_id",
        "employee_last_name",
        "employee_first_name",
        "employee_birthdate",
        "employee_hiredate",
        "employee_phone",
        "employee_fax",
        "employee_email",
    ]
]

# Change the column datatypes.
df_normalized_db_employee_table = df_normalized_db_employee_table.astype({
    "employee_id": "Int64",
    "employee_level_id": "Int64",
    "employee_title_id": "Int64",
    "address_id": "Int64",
    "manager_id": "Int64",
    "employee_last_name": "string",
    "employee_first_name": "string",
    "employee_birthdate": "datetime64[ns]",
    "employee_hiredate": "datetime64[ns]",
    "employee_phone": "string",
    "employee_fax": "string",
    "employee_email": "string",
})

# Inspect the normalized "employee" DataFrame.
df_normalized_db_employee_table.head(9)

,employee_id,employee_level_id,employee_title_id,address_id,manager_id,employee_last_name,employee_first_name,employee_birthdate,employee_hiredate,employee_phone,employee_fax,employee_email
0,1,5,1,1,9,Adams,Andrew,1962-02-18,2016-08-14,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,4,4,2,1,Edwards,Nancy,1958-12-08,2016-05-01,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,1,5,3,2,Peacock,Jane,1973-08-29,2017-04-01,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,1,5,4,2,Park,Margaret,1947-09-19,2017-05-03,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,1,5,5,2,Johnson,Steve,1965-03-03,2017-10-17,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com
5,6,3,2,6,1,Mitchell,Michael,1973-07-01,2016-10-17,+1 (403) 246-9887,+1 (403) 246-9899,michael@chinookcorp.com
6,7,2,3,7,6,King,Robert,1970-05-29,2017-01-02,+1 (403) 456-9986,+1 (403) 456-8485,robert@chinookcorp.com
7,8,2,3,8,6,Callahan,Laura,1968-01-09,2017-03-04,+1 (403) 467-3351,+1 (403) 467-8772,laura@chinookcorp.com
8,9,6,6,9,<NA>,Madan,Mohan,1961-01-26,2016-01-14,+1 (780) 428-9482,+1 (780) 428-3457,madan.mohan@chinookcorp.com


In [112]:
df_normalized_db_employee_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   employee_id          9 non-null      Int64         
 1   employee_level_id    9 non-null      Int64         
 2   employee_title_id    9 non-null      Int64         
 3   address_id           9 non-null      Int64         
 4   manager_id           8 non-null      Int64         
 5   employee_last_name   9 non-null      string        
 6   employee_first_name  9 non-null      string        
 7   employee_birthdate   9 non-null      datetime64[ns]
 8   employee_hiredate    9 non-null      datetime64[ns]
 9   employee_phone       9 non-null      string        
 10  employee_fax         9 non-null      string        
 11  employee_email       9 non-null      string        
dtypes: Int64(5), datetime64[ns](2), string(5)
memory usage: 1.6 KB


In [113]:
# As explained in the README file of the project in GitHub, we temporarily nullify all 
# "manager_id" values to avoid self‑referencing FOREIGN KEY violations during the migration process.
df_normalized_db_employee_table["manager_id"] = pd.NA

In [114]:
# Create the normalized "customer" DataFrame.

# Use the previously imported "customer" entity from the original database
# to copy some of the needed columns in the normalized "customer" DataFrame.
df_normalized_db_customer_table = df_original_db_customer_table[["customer_id", "support_rep_id", "last_name", "first_name", "company", "phone", "fax", "email", "address"]].rename(
    columns={
        "support_rep_id": "representative_id",
        "last_name": "customer_last_name",
        "first_name": "customer_first_name",
        "company": "customer_company",
        "phone": "customer_phone",
        "fax": "customer_fax",
        "email": "customer_email",
    }
)

In [115]:
df_normalized_db_customer_table.head()

,customer_id,representative_id,customer_last_name,customer_first_name,customer_company,customer_phone,customer_fax,customer_email,address
0,1,3,Gonçalves,Luís,Embraer - Empresa Brasileira de Aeronáutica S.A.,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,"Av. Brigadeiro Faria Lima, 2170"
1,2,5,Köhler,Leonie,NaN,+49 0711 2842222,NaN,leonekohler@surfeu.de,Theodor-Heuss-Straße 34
2,3,3,Tremblay,François,NaN,+1 (514) 721-4711,NaN,ftremblay@gmail.com,1498 rue Bélanger
3,4,4,Hansen,Bjørn,NaN,+47 22 44 22 22,NaN,bjorn.hansen@yahoo.no,Ullevålsveien 14
4,6,5,Holý,Helena,NaN,+420 2 4177 0449,NaN,hholy@gmail.com,Rilská 3174/6


In [116]:
# Use "merge()" to add the corresponding address IDs of each customer from the normalized "address" DataFrame.
df_normalized_db_customer_table["address_id"] = (
    df_normalized_db_customer_table.merge(
        df_normalized_db_address_table[["address_line_1", "address_id"]],
        left_on="address",
        right_on="address_line_1",
        how="left"
    )["address_id"]
)

In [117]:
df_normalized_db_customer_table.head()

,customer_id,representative_id,customer_last_name,customer_first_name,customer_company,customer_phone,customer_fax,customer_email,address,address_id
0,1,3,Gonçalves,Luís,Embraer - Empresa Brasileira de Aeronáutica S.A.,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,"Av. Brigadeiro Faria Lima, 2170",10
1,2,5,Köhler,Leonie,NaN,+49 0711 2842222,NaN,leonekohler@surfeu.de,Theodor-Heuss-Straße 34,11
2,3,3,Tremblay,François,NaN,+1 (514) 721-4711,NaN,ftremblay@gmail.com,1498 rue Bélanger,12
3,4,4,Hansen,Bjørn,NaN,+47 22 44 22 22,NaN,bjorn.hansen@yahoo.no,Ullevålsveien 14,13
4,6,5,Holý,Helena,NaN,+420 2 4177 0449,NaN,hholy@gmail.com,Rilská 3174/6,14


In [118]:
# Drop the "address" column.
df_normalized_db_customer_table = df_normalized_db_customer_table.drop("address", axis=1)

In [119]:
# Rearrange the order of the normalized "customer" DataFrame.

df_normalized_db_customer_table = df_normalized_db_customer_table[
    [
        "customer_id",
        "address_id",
        "representative_id",
        "customer_last_name",
        "customer_first_name",
        "customer_company",
        "customer_phone",
        "customer_fax",
        "customer_email",
    ]
]

# Change the column datatypes.
df_normalized_db_customer_table = df_normalized_db_customer_table.astype({
    "customer_id": "Int64",
    "address_id": "Int64",
    "representative_id": "Int64",
    "customer_last_name": "string",
    "customer_first_name": "string",
    "customer_company": "string",
    "customer_phone": "string",
    "customer_fax": "string",
    "customer_email": "string",
})

In [120]:
# Normalize the missing phone number.
# The original dataset contains one customer with a <NA> phone value.
# Since the new schema requires the "customer_phone" column to be NOT NULL, we assign a placeholder value.
# For more details, check the "Phone Number Normalization During Migration" section of the README file in GitHub.
df_normalized_db_customer_table.loc[df_normalized_db_customer_table["customer_phone"].isna(), "customer_phone"] = "0000000000"

In [121]:
df_normalized_db_customer_table.head()

,customer_id,address_id,representative_id,customer_last_name,customer_first_name,customer_company,customer_phone,customer_fax,customer_email
0,1,10,3,Gonçalves,Luís,Embraer - Empresa Brasileira de Aeronáutica S.A.,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br
1,2,11,5,Köhler,Leonie,<NA>,+49 0711 2842222,<NA>,leonekohler@surfeu.de
2,3,12,3,Tremblay,François,<NA>,+1 (514) 721-4711,<NA>,ftremblay@gmail.com
3,4,13,4,Hansen,Bjørn,<NA>,+47 22 44 22 22,<NA>,bjorn.hansen@yahoo.no
4,6,14,5,Holý,Helena,<NA>,+420 2 4177 0449,<NA>,hholy@gmail.com


In [122]:
df_normalized_db_customer_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          59 non-null     Int64 
 1   address_id           59 non-null     Int64 
 2   representative_id    59 non-null     Int64 
 3   customer_last_name   59 non-null     string
 4   customer_first_name  59 non-null     string
 5   customer_company     10 non-null     string
 6   customer_phone       59 non-null     string
 7   customer_fax         12 non-null     string
 8   customer_email       59 non-null     string
dtypes: Int64(3), string(6)
memory usage: 7.8 KB


**At this point we have finalziled the normalized tables for the following entities:**<br>

***genre (df_normalized_db_genre_table)**<br>
***genre_track (df_normalized_db_genre_track_table)**<br>
***media_type (df_normalized_db_media_type_table)**<br>
***composer (df_normalized_db_composer_table)**<br>
***composer_track (df_normalized_db_composer_track_table)**<br>
***album (df_normalized_db_album_table)**<br>
***artist (df_normalized_db_artist_table)**<br>
***artist_album (df_normalized_db_artist_album_table)**<br>
***album_track (df_normalized_db_album_track_table)**<br>
***track (df_normalized_db_track_table)**<br>
***playlist (df_normalized_db_playlist_table)**<br>
***playlist_track (df_normalized_db_playlist_track_table)**<br>
***invoice (df_normalized_db_invoice_table)**<br>
***invoice_line (df_normalized_db_invoice_line_table)**<br>
***country (df_normalized_db_country_table)**<br>
***postal_zip_code (df_normalized_db_postal_zip_code_table)**<br>
***administrative_division (df_normalized_db_administrative_division_table)**<br>
***city (df_normalized_db_city_table)**<br>
***address (df_normalized_db_address_table)**<br>
***employee_title (df_normalized_db_employee_title_table)**<br>
***employee_level (df_normalized_db_employee_level_table)**<br>
***employee (df_normalized_db_employee_table)**<br>
***customer (df_normalized_db_customer_table)**<br>

In [123]:
# Create a dictionary that holds the names of all created DataFrames.
# This dictionary will be later used to clean and bulk-insert each DataFrame into the normalized Database.
df_dictionary = {
    "genre": df_normalized_db_genre_table,
    "genre_track": df_normalized_db_genre_track_table,
    "media_type": df_normalized_db_media_type_table,
    "composer": df_normalized_db_composer_table,
    "composer_track": df_normalized_db_composer_track_table,
    "album": df_normalized_db_album_table,
    "artist": df_normalized_db_artist_table,
    "artist_album": df_normalized_db_artist_album_table,
    "album_track": df_normalized_db_album_track_table,
    "track": df_normalized_db_track_table,
    "playlist": df_normalized_db_playlist_table,
    "playlist_track": df_normalized_db_playlist_track_table,
    "invoice": df_normalized_db_invoice_table,
    "invoice_line": df_normalized_db_invoice_line_table,
    "country": df_normalized_db_country_table,
    "postal_zip_code": df_normalized_db_postal_zip_code_table,
    "administrative_division": df_normalized_db_administrative_division_table,
    "city": df_normalized_db_city_table,
    "address": df_normalized_db_address_table,
    "employee_title": df_normalized_db_employee_title_table,
    "employee_level": df_normalized_db_employee_level_table,
    "employee": df_normalized_db_employee_table,
    "customer": df_normalized_db_customer_table,
}

**Move to define the ORM classes.**

In [124]:
# In SQLAlchemy, we can define a table schema using the declarative_base class, which all mapped classes will inherit from.
Base = declarative_base()

In [125]:
# Create the database tables
class Album(Base):
    __tablename__ = "album"

    album_id = Column(Integer, primary_key=True)
    album_title = Column(String(255), nullable=False)

    artist_albums = relationship(
        "ArtistAlbum",
        back_populates="album",
        cascade="save-update, merge",
        passive_deletes=True
    )
    # The ".artists" relationship returns a list of Artist objects linked to an album via the "artist_album" junction table.
    artists = relationship(
        "Artist",
        secondary="artist_album",
        viewonly=True
    )

    album_tracks = relationship(
        "AlbumTrack",
        back_populates="album",
        cascade="save-update, merge",
        passive_deletes=True
    )
    # The ".tracks" relationship returns a list of Track objects linked to an album via the "album_track" junction table.
    tracks = relationship(
        "Track",
        secondary="album_track",
        viewonly=True
    )

    __table_args__ = (
        Index("ix_album_album_title", "album_title"),
        CheckConstraint("trim(album_title) <> ''", name="ck_album_title_not_whitespace"),
    )


class Artist(Base):
    __tablename__ = "artist"

    artist_id = Column(Integer, primary_key=True)
    artist_name = Column(String(255), nullable=False)

    artist_albums = relationship(
        "ArtistAlbum",
        back_populates="artist",
        cascade="save-update, merge",
        passive_deletes=True
    )

    # The ".albums" relationship returns a list of Album objects linked to an artist via the "artist_album" junction table.
    albums = relationship(
        "Album",
        secondary="artist_album",
        viewonly=True
    )

    __table_args__ = (
        Index("ix_artist_artist_name", "artist_name"),
        CheckConstraint("trim(artist_name) <> ''", name="ck_artist_name_not_whitespace"),
    )

class ArtistAlbum(Base):
    __tablename__ = "artist_album"

    artist_album_id = Column(Integer, primary_key=True)
    artist_id = Column(
        Integer,
        ForeignKey("artist.artist_id", ondelete="RESTRICT"),
        nullable=False
    )
    album_id = Column(
        Integer,
        ForeignKey("album.album_id", ondelete="RESTRICT"),
        nullable=False
    )

    artist = relationship(
        "Artist",
        back_populates="artist_albums",
        passive_deletes=True
    )

    album = relationship(
        "Album",
        back_populates="artist_albums",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_artist_album_artist_id", "artist_id"),
        Index("ix_artist_album_album_id", "album_id"),
        Index("ix_artist_album_pair", "artist_id", "album_id", unique=True),
    )

class Composer(Base):
    __tablename__ = "composer"

    composer_id = Column(Integer, primary_key=True)
    composer_name = Column(String(255), nullable=False)

    composer_tracks = relationship(
        "ComposerTrack",
        back_populates="composer",
        cascade="save-update, merge",
        passive_deletes=True
    )

    tracks = relationship(
        "Track",
        secondary="composer_track",
        viewonly=True
    )

    __table_args__ = (
        Index("ix_composer_composer_name", "composer_name"),
        CheckConstraint(
            "composer_name IS NULL OR trim(composer_name) <> ''",
            name="ck_composer_name_not_whitespace"
        ),
    )

class Genre(Base):
    __tablename__ = "genre"

    genre_id = Column(Integer, primary_key=True)
    genre_name = Column(String(255), nullable=False)

    genre_tracks = relationship(
        "GenreTrack",
        back_populates="genre",
        cascade="save-update, merge",
        passive_deletes=True
    )

    tracks = relationship(
        "Track",
        secondary="genre_track",
        viewonly=True
    )

    __table_args__ = (
        UniqueConstraint("genre_name", name="uq_genre_name"),
        CheckConstraint("trim(genre_name) <> ''", name="ck_genre_name_not_whitespace"),
    )

class MediaType(Base):
    __tablename__ = "media_type"

    media_type_id = Column(Integer, primary_key=True)
    media_type_name = Column(String(255), nullable=False)

    tracks = relationship(
        "Track",
        back_populates="media_type",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        UniqueConstraint("media_type_name", name="uq_media_type_name"),
        CheckConstraint("trim(media_type_name) <> ''", name="ck_media_type_name_not_whitespace"),
    )

class Playlist(Base):
    __tablename__ = "playlist"

    playlist_id = Column(Integer, primary_key=True)
    playlist_name = Column(String(255), nullable=False)

    playlist_tracks = relationship(
        "PlaylistTrack",
        back_populates="playlist",
        cascade="save-update, merge",
        passive_deletes=True
    )

    tracks = relationship(
        "Track",
        secondary="playlist_track",
        viewonly=True
    )

    __table_args__ = (
        CheckConstraint("trim(playlist_name) <> ''", name="ck_playlist_name_not_whitespace"),
    )

class Track(Base):
    __tablename__ = "track"

    track_id = Column(BigInteger, primary_key=True)

    media_type_id = Column(
        Integer,
        ForeignKey("media_type.media_type_id", ondelete="RESTRICT"),
        nullable=False
    )

    track_name = Column(String(255), nullable=False)
    duration_milliseconds = Column(Integer, nullable=False)
    size_bytes = Column(Integer, nullable=False)
    unit_price_cents = Column(Integer, server_default=text("0"), nullable=False)

    media_type = relationship(
        "MediaType",
        back_populates="tracks",
        passive_deletes=True
    )

    composer_tracks = relationship(
        "ComposerTrack",
        back_populates="track",
        cascade="save-update, merge",
        passive_deletes=True
    )

    genre_tracks = relationship(
        "GenreTrack",
        back_populates="track",
        cascade="save-update, merge",
        passive_deletes=True
    )

    playlist_tracks = relationship(
        "PlaylistTrack",
        back_populates="track",
        cascade="save-update, merge",
        passive_deletes=True
    )

    album_tracks = relationship(
        "AlbumTrack",
        back_populates="track",
        cascade="save-update, merge",
        passive_deletes=True
    )

    composers = relationship(
        "Composer",
        secondary="composer_track",
        viewonly=True
    )

    genres = relationship(
        "Genre",
        secondary="genre_track",
        viewonly=True
    )

    playlists = relationship(
        "Playlist",
        secondary="playlist_track",
        viewonly=True
    )

    albums = relationship(
        "Album",
        secondary="album_track",
        viewonly=True
    )

    invoice_lines = relationship(
        "InvoiceLine",
        back_populates="track",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_track_media_type_id", "media_type_id"),
        Index("ix_track_track_name", "track_name"),
        Index("ix_track_unit_price_cents", "unit_price_cents"),
        CheckConstraint("trim(track_name) <> ''", name="ck_track_name_not_whitespace"),
        CheckConstraint("duration_milliseconds > 0", name="ck_track_duration_positive"),
        CheckConstraint("size_bytes > 0", name="ck_track_size_positive"),
        CheckConstraint("unit_price_cents >= 0", name="ck_track_price_non_negative"),
    )

class AlbumTrack(Base):
    __tablename__ = "album_track"

    album_track_id = Column(Integer, primary_key=True)
    album_id = Column(
        Integer,
        ForeignKey("album.album_id", ondelete="RESTRICT"),
        nullable=False
    )
    track_id = Column(
        BigInteger,
        ForeignKey("track.track_id", ondelete="RESTRICT"),
        nullable=False
    )

    album = relationship(
        "Album",
        back_populates="album_tracks",
        passive_deletes=True
    )

    track = relationship(
        "Track",
        back_populates="album_tracks",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_album_track_album_id", "album_id"),
        Index("ix_album_track_track_id", "track_id"),
        Index("ix_album_track_pair", "album_id", "track_id", unique=True),
    )

class ComposerTrack(Base):
    __tablename__ = "composer_track"

    composer_track_id = Column(Integer, primary_key=True)
    composer_id = Column(
        Integer,
        ForeignKey("composer.composer_id", ondelete="RESTRICT"),
        nullable=False
    )
    track_id = Column(
        BigInteger,
        ForeignKey("track.track_id", ondelete="RESTRICT"),
        nullable=False
    )

    composer = relationship(
        "Composer",
        back_populates="composer_tracks",
        passive_deletes=True
    )

    track = relationship(
        "Track",
        back_populates="composer_tracks",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_composer_track_composer_id", "composer_id"),
        Index("ix_composer_track_track_id", "track_id"),
        Index("ix_composer_track_pair", "composer_id", "track_id", unique=True),
    )

class GenreTrack(Base):
    __tablename__ = "genre_track"

    genre_track_id = Column(Integer, primary_key=True)
    genre_id = Column(
        Integer,
        ForeignKey("genre.genre_id", ondelete="RESTRICT"),
        nullable=False
    )
    track_id = Column(
        BigInteger,
        ForeignKey("track.track_id", ondelete="RESTRICT"),
        nullable=False
    )

    genre = relationship(
        "Genre",
        back_populates="genre_tracks",
        passive_deletes=True
    )

    track = relationship(
        "Track",
        back_populates="genre_tracks",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_genre_track_genre_id", "genre_id"),
        Index("ix_genre_track_track_id", "track_id"),
        Index("ix_genre_track_pair", "genre_id", "track_id", unique=True),
    )

class PlaylistTrack(Base):
    __tablename__ = "playlist_track"

    playlist_track_id = Column(Integer, primary_key=True)
    playlist_id = Column(
        Integer,
        ForeignKey("playlist.playlist_id", ondelete="RESTRICT"),
        nullable=False
    )
    track_id = Column(
        BigInteger,
        ForeignKey("track.track_id", ondelete="RESTRICT"),
        nullable=False
    )

    playlist = relationship(
        "Playlist",
        back_populates="playlist_tracks",
        passive_deletes=True
    )

    track = relationship(
        "Track",
        back_populates="playlist_tracks",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_playlist_track_playlist_id", "playlist_id"),
        Index("ix_playlist_track_track_id", "track_id"),
        Index("ix_playlist_track_pair", "playlist_id", "track_id", unique=True),
    )

class Country(Base):
    __tablename__ = "country"

    country_id = Column(Integer, primary_key=True)
    country_name = Column(String(255), nullable=False)
    country_iso_code = Column(String(3), nullable=False)

    postal_zip_codes = relationship(
        "PostalZipCode",
        back_populates="country",
        cascade="save-update, merge",
        passive_deletes=True
    )

    administrative_divisions = relationship(
        "AdministrativeDivision",
        back_populates="country",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("idx_country_country_name", "country_name"),
        UniqueConstraint("country_iso_code", name="country_iso_code"),
        CheckConstraint("trim(country_name) <> ''", name="ck_country_name_not_whitespace"),
        CheckConstraint("char_length(country_iso_code) IN (2, 3)", name="ck_country_iso_code_length"),
        CheckConstraint("country_iso_code = upper(country_iso_code)", name="ck_country_iso_code_upper"),
    )

class PostalZipCode(Base):
    __tablename__ = "postal_zip_code"

    postal_zip_code_id = Column(Integer, primary_key=True)
    country_id = Column(
        Integer,
        ForeignKey("country.country_id", ondelete="RESTRICT"),
        nullable=False
    )
    code = Column(String(12), nullable=True)

    country = relationship(
        "Country",
        back_populates="postal_zip_codes",
        passive_deletes=True
    )

    addresses = relationship(
        "Address",
        back_populates="postal_zip_code",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("idx_postal_zip_code_code", "code"),
        UniqueConstraint("country_id", "code", name="uq_country_postal_zip_code"),
        CheckConstraint("trim(code) <> ''", name="ck_code_not_whitespace"),
    )

class AdministrativeDivision(Base):
    __tablename__ = "administrative_division"

    administrative_division_id = Column(Integer, primary_key=True)
    country_id = Column(
        Integer,
        ForeignKey("country.country_id", ondelete="RESTRICT"),
        nullable=False
    )
    division_name = Column(String(255), nullable=False)
    division_type = Column(String(255), nullable=False)

    country = relationship(
        "Country",
        back_populates="administrative_divisions",
        passive_deletes=True
    )

    cities = relationship(
        "City",
        back_populates="administrative_division",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_administrative_division_division_name", "division_name"),
        UniqueConstraint("country_id", "division_name", name="uq_country_division_name"),
        CheckConstraint("trim(division_name) <> ''", name="ck_division_name_not_whitespace"),
        CheckConstraint("trim(division_type) <> ''", name="ck_division_type_not_whitespace"),
    )

class City(Base):
    __tablename__ = "city"

    city_id = Column(Integer, primary_key=True)
    administrative_division_id = Column(
        Integer,
        ForeignKey("administrative_division.administrative_division_id", ondelete="RESTRICT"),
        nullable=False
    )
    city_name = Column(String(255), nullable=False)

    administrative_division = relationship(
        "AdministrativeDivision",
        back_populates="cities",
        passive_deletes=True
    )

    addresses = relationship(
        "Address",
        back_populates="city",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_city_city_name", "city_name"),
        UniqueConstraint("administrative_division_id", "city_name", name="uq_division_city_name"),
        CheckConstraint("trim(city_name) <> ''", name="ck_city_name_not_whitespace"),
    )

class Address(Base):
    __tablename__ = "address"

    address_id = Column(Integer, primary_key=True)
    city_id = Column(
        Integer,
        ForeignKey("city.city_id", ondelete="RESTRICT"),
        nullable=False
    )
    postal_zip_code_id = Column(
        Integer,
        ForeignKey("postal_zip_code.postal_zip_code_id", ondelete="RESTRICT"),
        nullable=False
    )
    address_line_1 = Column(String(255), nullable=False)
    address_line_2 = Column(String(255))

    city = relationship(
        "City",
        back_populates="addresses",
        passive_deletes=True
    )

    postal_zip_code = relationship(
        "PostalZipCode",
        back_populates="addresses",
        passive_deletes=True
    )

    customers = relationship(
        "Customer",
        back_populates="address",
        cascade="save-update, merge",
        passive_deletes=True
    )

    employees = relationship(
        "Employee",
        back_populates="address",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_address_city_id", "city_id"),
        Index("ix_address_postal_zip_code_id", "postal_zip_code_id"),
        CheckConstraint("trim(address_line_1) <> ''", name="ck_address_line_1_not_whitespace"),
        CheckConstraint("trim(address_line_2) IS NULL OR trim(address_line_2) <> ''", name="ck_address_line_2_not_whitespace_if_present"),
    )

class Customer(Base):
    __tablename__ = "customer"

    customer_id = Column(Integer, primary_key=True)
    address_id = Column(
        Integer,
        ForeignKey("address.address_id", ondelete="RESTRICT"),
        nullable=True
    )
    representative_id = Column(
        Integer,
        ForeignKey("employee.employee_id", ondelete="RESTRICT"),
        nullable=False
    )
    customer_last_name = Column(String(255), nullable=False)
    customer_first_name = Column(String(255), nullable=False)
    customer_company = Column(String(255))
    customer_phone = Column(String(255), nullable=False)
    customer_fax = Column(String(255))
    customer_email = Column(String(255), nullable=False)

    address = relationship(
        "Address",
        back_populates="customers",
        passive_deletes=True
    )

    representative = relationship(
        "Employee",
        back_populates="customers",
        passive_deletes=True
    )

    invoices = relationship(
        "Invoice",
        back_populates="customer",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_customer_address_id", "address_id"),
        Index("ix_customer_representative_id", "representative_id"),
        UniqueConstraint("customer_email", name="uq_customer_email"),
        CheckConstraint("trim(customer_last_name) <> ''", name="ck_customer_last_name_not_whitespace"),
        CheckConstraint("trim(customer_first_name) <> ''", name="ck_customer_first_name_not_whitespace"),
        CheckConstraint("trim(customer_company) IS NULL OR trim(customer_company) <> ''", name="ck_customer_company_not_whitespace_if_present"),
        CheckConstraint("trim(customer_phone) <> ''", name="ck_customer_phone_not_whitespace"),
        CheckConstraint("trim(customer_fax) IS NULL OR trim(customer_fax) <> ''", name="ck_customer_fax_not_whitespace_if_present"),
        CheckConstraint("trim(customer_email) <> ''", name="ck_customer_email_not_whitespace"),
    )

class EmployeeLevel(Base):
    __tablename__ = "employee_level"

    employee_level_id = Column(Integer, primary_key=True)
    employee_level_code = Column(String(2), nullable=False)

    employees = relationship(
        "Employee",
        back_populates="level",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        UniqueConstraint("employee_level_code", name="uq_employee_level_code"),
        CheckConstraint("trim(employee_level_code) <> ''", name="ck_employee_level_code_not_whitespace"),
        CheckConstraint("char_length(employee_level_code) = 2", name="ck_employee_level_code_length"),
    )
            
class EmployeeTitle(Base):
    __tablename__ = "employee_title"

    employee_title_id = Column(Integer, primary_key=True)
    employee_title_name = Column(String(255), nullable=False)

    employees = relationship(
        "Employee",
        back_populates="title",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        UniqueConstraint("employee_title_name", name="uq_employee_title_name"),
        CheckConstraint("trim(employee_title_name) <> ''", name="ck_employee_title_name_not_whitespace"),
    )

class Employee(Base):
    __tablename__ = "employee"

    employee_id = Column(Integer, primary_key=True)
    employee_level_id = Column(
        Integer,
        ForeignKey("employee_level.employee_level_id", ondelete="RESTRICT"),
        nullable=False
    )
    employee_title_id = Column(
        Integer,
        ForeignKey("employee_title.employee_title_id", ondelete="RESTRICT"),
        nullable=False
    )
    address_id = Column(
        Integer,
        ForeignKey("address.address_id", ondelete="RESTRICT"),
        nullable=False
    )
    manager_id = Column(
        Integer,
        ForeignKey("employee.employee_id", ondelete="RESTRICT"),
        nullable=True
    )
    employee_last_name = Column(String(255), nullable=False)
    employee_first_name = Column(String(255), nullable=False)
    employee_birthdate = Column(Date, nullable=False)
    employee_hiredate = Column(Date, nullable=False)
    employee_phone = Column(String(255), nullable=False)
    employee_fax = Column(String(255))
    employee_email = Column(String(255), nullable=False)

    level = relationship(
        "EmployeeLevel",
        back_populates="employees",
        passive_deletes=True
    )

    title = relationship(
        "EmployeeTitle",
        back_populates="employees",
        passive_deletes=True
    )

    address = relationship(
        "Address",
        back_populates="employees",
        passive_deletes=True
    )

    manager = relationship(
        "Employee",
        remote_side=[employee_id],
        backref="subordinates",
        passive_deletes=True
    )

    customers = relationship(
        "Customer",
        back_populates="representative",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_employee_employee_level_id", "employee_level_id"),
        Index("ix_employee_employee_title_id", "employee_title_id"),
        Index("ix_employee_last_name", "employee_last_name"),
        Index("ix_employee_address_id", "address_id"),
        Index("ix_employee_manager_id", "manager_id"),
        UniqueConstraint("employee_email", name="uq_employee_email"),
        CheckConstraint("trim(employee_last_name) <> ''", name="ck_employee_last_name_not_whitespace"),
        CheckConstraint("trim(employee_first_name) <> ''", name="ck_employee_first_name_not_whitespace"),
        CheckConstraint("employee_birthdate < employee_hiredate", name="ck_employee_birth_before_hire"),
        CheckConstraint("trim(employee_phone) <> ''", name="ck_employee_phone_not_whitespace"),
        CheckConstraint("trim(employee_fax) IS NULL OR trim(employee_fax) <> ''", name="ck_employee_fax_not_whitespace_if_present"),
        CheckConstraint("trim(employee_email) <> ''", name="ck_employee_email_not_whitespace"),
    )

class Invoice(Base):
    __tablename__ = "invoice"

    invoice_id = Column(Integer, primary_key=True)
    customer_id = Column(
        Integer,
        ForeignKey("customer.customer_id", ondelete="RESTRICT"),
        nullable=False
    )
    invoice_datetime = Column(TIMESTAMP(timezone=True), nullable=False)
    billing_address = Column(String(255), nullable=False)
    billing_city = Column(String(255), nullable=False)
    billing_administrative_division = Column(String(255))
    billing_country = Column(String(255), nullable=False)
    billing_postal = Column(String(255))
    invoice_total_cents = Column(Integer, nullable=False)

    customer = relationship(
        "Customer",
        back_populates="invoices",
        passive_deletes=True
    )

    invoice_lines = relationship(
        "InvoiceLine",
        back_populates="invoice",
        cascade="save-update, merge",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_invoice_customer_id", "customer_id"),
        Index("ix_invoice_invoice_datetime", "invoice_datetime"),
        CheckConstraint("trim(billing_address) <> ''", name="ck_billing_address_not_whitespace"),
        CheckConstraint("trim(billing_city) <> ''", name="ck_billing_city_not_whitespace"),
        CheckConstraint("billing_administrative_division IS NULL OR trim(billing_administrative_division) <> ''", name="ck_billing_administrative_division_not_whitespace"),
        CheckConstraint("trim(billing_country) <> ''", name="ck_billing_country_not_whitespace"),
        CheckConstraint("trim(billing_postal) <> ''", name="ck_billing_postal_not_whitespace"),
        CheckConstraint("invoice_total_cents >= 0", name="ck_invoice_total_cents_non_negative"),
    )

class InvoiceLine(Base):
    __tablename__ = "invoice_line"

    invoice_line_id = Column(Integer, primary_key=True)
    invoice_id = Column(
        Integer,
        ForeignKey("invoice.invoice_id", ondelete="RESTRICT"),
        nullable=False
    )
    track_id = Column(
        BigInteger,
        ForeignKey("track.track_id", ondelete="RESTRICT"),
        nullable=False
    )
    unit_price_cents = Column(Integer, nullable=False)
    quantity = Column(Integer, nullable=False)

    invoice = relationship(
        "Invoice",
        back_populates="invoice_lines",
        passive_deletes=True
    )

    track = relationship(
        "Track",
        back_populates="invoice_lines",
        passive_deletes=True
    )

    __table_args__ = (
        Index("ix_invoice_line_invoice_id", "invoice_id"),
        Index("ix_invoice_line_track_id", "track_id"),
        CheckConstraint("unit_price_cents >= 0", name="ck_unit_price_cents_non_negative"),
        CheckConstraint("quantity > 0", name="ck_quantity_positive"),
    )

In [126]:
# Drop existing tables (only during development)

# Base.metadata.drop_all(engine_normalized_db)

# Use the following commands in pdAdmin to delete the tables and reset the schema.
# DROP SCHEMA public CASCADE;
# CREATE SCHEMA public;

# GRANT ALL ON SCHEMA public TO postgres;
# GRANT ALL ON SCHEMA public TO public;

In [127]:
# Create the normalized database entities in PostgreSQL.
Base.metadata.create_all(engine_normalized_db)

In [128]:
# Create a custom function that will replace missing values (pd.NA, np.nan, NaT) with None in a provided DataFrame.
# This will be used, before inserting each DataFrame into the corresponding table of the normalized database.
def clean_dataframe(df):
    df = df.where(df.notna(), None)
    return df

In [129]:
# Create a mapping dictionary that will help us align each ORM class with its corresponding database table.
models = {
    "genre": Genre,
    "genre_track": GenreTrack,
    "media_type": MediaType,
    "composer": Composer,
    "composer_track": ComposerTrack,
    "album": Album,
    "artist": Artist,
    "artist_album": ArtistAlbum,
    "album_track": AlbumTrack,
    "track": Track,
    "playlist": Playlist,
    "playlist_track": PlaylistTrack,
    "invoice": Invoice,
    "invoice_line": InvoiceLine,
    "country": Country,
    "postal_zip_code": PostalZipCode,
    "administrative_division": AdministrativeDivision,
    "city": City,
    "address": Address,
    "employee_title": EmployeeTitle,
    "employee_level": EmployeeLevel,
    "employee": Employee,
    "customer": Customer,
}

In [130]:
# Create a list which ensures that we respect FOREIGN KEY dependencies, while inserting the data into the normalized database.
insertion_order = [
    "album",                    # Root table.
    "artist",                   # Root table.
    "composer",                 # Root table.
    "genre",                    # Root table.
    "media_type",               # Root table.
    "playlist",                 # Root table.
    "country",                  # Root table.
    "employee_level",           # Root table.
    "employee_title",           # Root table.
    "track",                    # Depends on table "media_type".
    "postal_zip_code",          # Depends on table "country".
    "administrative_division",  # Depends on table "country".
    "city",                     # Depends on table "administrative_division".
    "address",                  # Depends on tables "city" and "postal_zip_code".
    "employee",                 # Depends on tables "address", "employee_level" and "employee_title".
    "customer",                 # Depends on tables "address" and "employee".
    "invoice",                  # Depends on table "customer".
    "invoice_line",             # Depends on tables "invoice" and "track".
    "artist_album",             # Junction table of "album" and "artist".
    "album_track",              # Junction table of "album" and "track".
    "composer_track",           # Junction table of "composer" and "track".
    "genre_track",              # Junction table of "genre" and "track".
    "playlist_track",           # Junction table of "playlist" and "track".
]

In [131]:
# Create the custom function that loads the data from the DataFrames into the normalized PostgreSQL database.
def bulk_insert(model, rows, session):
    if not rows:
        return  # Nothing to insert.

    stmt = insert(model) # Crete an INSERT statement based on the ORM classes.
    session.execute(stmt, rows)
    session.commit()

In [132]:
# Start the migration process, using a try-except block, that informs us when an insertion succeeds or fails.
try:
    # Create a flag that ensures updates to the "employee" table are implemented, only if the migration process was successful.
    all_inserted = True

    # Using the following loop we iterate over the "insertion_order" list. 
    for name in insertion_order:
        try:
            # Retrieve the DataFrame for the current table and clean it.
            df = clean_dataframe(df_dictionary[name])
            # Convert the cleaned DataFrame into a list of dictionaries.
            rows = df.to_dict(orient="records")
            # Insert all rows into the database using the correct ORM model.
            bulk_insert(models[name], rows, session)
            print(f"Inserted table: {name}")
        except Exception as e:
            print(f"Failed to insert table: {name}")
            print(e)
            all_inserted = False
            break

    # Update the "employee" table, if all inserts succeeded.
    if all_inserted:
        try:
            # Mapping each "employee_id" to its corresponding "manager_id"
            updates = {
                1: 9,
                2: 1,
                3: 2,
                4: 2,
                5: 2,
                6: 1,
                7: 6,
                8: 6,
            }

            Employee = models["employee"]

            for emp_id, mgr_id in updates.items():
                session.query(Employee).filter_by(employee_id=emp_id).update(
                    {Employee.manager_id: mgr_id},
                    synchronize_session=False
                )

            session.commit()
            print("Manager IDs updated successfully.")

        except Exception as e:
            session.rollback()
            print("Failed to update manager IDs.")
            print(e)

finally:
    session.close()
    print("Session closed.")

Inserted table: album
Inserted table: artist
Inserted table: composer
Inserted table: genre
Inserted table: media_type
Inserted table: playlist
Inserted table: country
Inserted table: employee_level
Inserted table: employee_title
Inserted table: track
Inserted table: postal_zip_code
Inserted table: administrative_division
Inserted table: city
Inserted table: address
Inserted table: employee
Inserted table: customer
Inserted table: invoice
Inserted table: invoice_line
Inserted table: artist_album
Inserted table: album_track
Inserted table: composer_track
Inserted table: genre_track
Inserted table: playlist_track
Manager IDs updated successfully.
Session closed.
